# 04_session_K_caformer_f2_f3

**Accelerator: GPU T4 x2, Internet ON.** CAFormer-S18 reader folds 2 and 3, one per GPU. Each GPU runs its jobs as separate processes with the exact single-GPU recipe; the notebook prints progress every two minutes and fails loudly if any job fails.

Attach the competition, both cache datasets made from the 02 shard outputs (knee-cache-v3-shard0/1, latest version: 18 files of ~1 GB each), the four public report-label datasets, your reader_labels dataset and the 01_labels output (compi-label-generation, for the Flash-Lite labels). Inputs are detected automatically.

Outputs land in one folder per family (`reader_convnext/`, `slot_dinov2/`, `radimagenet/`); attach this whole output to 05_submit.ipynb.

In [ ]:
# Training requires Internet for pretrained weights and this small split helper.
import importlib.util, subprocess, sys
if importlib.util.find_spec('iterstrat') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','iterative-stratification==0.1.9'])
# Keep the preinstalled CUDA torch/torchvision stack; match inference's timm.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm==1.0.29'])

In [ ]:
# GPU -> jobs run in order on that GPU: (family preset, folds).
GPU_JOBS = {
    0: [('reader_caformer', [2])],
    1: [('reader_caformer', [3])],
}
# Interrupted session: attach its saved output and set this to that output folder;
# folds with fold_N.pt and last_fold_N.pt resume at their next epoch.
RESUME_DIR = ""


In [ ]:
# Training code for each job (generated from the shared sources).
TRAIN_JOB_SOURCE = 'import os, gc, copy, math, time, json, random, warnings\nwarnings.filterwarnings("ignore")\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom tqdm.auto import tqdm\n\n# ---- inputs ------------------------------------------------------------------\n"""Bounded discovery of attached Kaggle artifacts; embedded in launch notebooks."""\nimport fnmatch, os\nfrom pathlib import Path\n\n\nDICOM_DIRS = (\'train_series\', \'test_series\')   # competition image trees: never listed\n\n\ndef attached_files(name, root=\'/kaggle/input\', max_depth=5):\n    # Kaggle mounts datasets and notebook outputs at different depths; artifacts live in\n    # folders 1-5 levels below the root. The DICOM trees are pruned: listing them took\n    # ~2 min on local disk and over an hour on the TPU VM\'s network-mounted inputs.\n    root = Path(root)\n    found = set()\n    for path, dirs, files in os.walk(root):\n        depth = len(Path(path).relative_to(root).parts)\n        dirs[:] = [d for d in dirs if d not in DICOM_DIRS] if depth < max_depth else []\n        if depth >= 1:\n            found.update(Path(path) / f for f in fnmatch.filter(files, name) if not f.startswith(\'.\'))\n    return sorted(found)\n\n\ndef one_input(name, override=\'\', root=\'/kaggle/input\'):\n    if override:\n        path = Path(override)\n        if not path.is_file():\n            raise FileNotFoundError(f\'Configured input does not exist: {path}\')\n        return str(path)\n    matches = attached_files(name, root)\n    if len(matches) != 1:\n        raise RuntimeError(f\'Attach exactly one intended {name} input or set its path explicitly. \'\n                           f\'Found: {[str(p) for p in matches]}\')\n    print(f\'{name}: {matches[0]}\')\n    return str(matches[0])\n\n\ndef fold_inputs(expected, root=\'/kaggle/input\', per_fold=1):\n    # per_fold > 1 when several training runs (e.g. DINOv2 and DINOv3 arms) are attached.\n    paths = []\n    for fold in expected:\n        matches = attached_files(f\'fold_{fold}.pt\', root)\n        if len(matches) != per_fold:\n            raise RuntimeError(f\'Attach exactly {per_fold} fold_{fold}.pt input(s) \'\n                               f\'({"exactly one" if per_fold == 1 else "one per training run"}). \'\n                               f\'Found: {[str(p) for p in matches]}\')\n        paths.extend(str(p) for p in matches)\n    for path in paths: print(f\'checkpoint: {path}\')\n    return paths\n\n\ndef checkpoint_inputs(root=\'/kaggle/input\'):\n    # Every attached fold_N.pt is an ensemble member; families are resolved from each\n    # checkpoint\'s recipe, and duplicates of one family-fold are rejected by the caller.\n    paths = [str(p) for p in attached_files(\'fold_*.pt\', root)]\n    if not paths:\n        raise RuntimeError(\'Attach at least one training output containing fold_N.pt\')\n    for path in paths: print(f\'checkpoint: {path}\')\n    return paths\n\n# Cache shards and label tables are discovered under /kaggle/input. Set CACHE_DIR to a\n# comma-separated list of shard folders only if the discovery is ambiguous.\nINPUT_ROOT = os.environ.get("KNEE_INPUT_ROOT", "/kaggle/input")   # other hosts: where inputs live\nCACHE_DIR = \',\'.join(str(p.parent) for p in attached_files(\'cache_meta.json\', INPUT_ROOT))\nassert CACHE_DIR, \'Attach the 02_cache shard outputs\'\nLABEL_ROOT = INPUT_ROOT\nCOMP_DIR  = os.environ.get("KNEE_COMP_DIR") or next(\n    (d for d in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection")\n     if os.path.isdir(d)), "/kaggle/input/competitions/rsna-knee-abnormality-detection")\nOUT       = os.environ.get("KNEE_OUT", "/kaggle/working")   # one folder per family in a session\n# -----------------------------------------------------------------------------\n# Several cache shards may be attached; a comma-separated list is accepted too.\nCACHE_DIRS = [d for d in CACHE_DIR.split(\',\') if d]\n\nLABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",\n              "PF OA","Effusion","Synovitis","Baker\'s","Contusion","Fracture"]\n\n_metas = [json.load(open(os.path.join(d, \'cache_meta.json\'))) for d in CACHE_DIRS]\n_recipe_only = lambda m: {k: v for k, v in m.items() if k not in (\'shard\', \'n_shards\')}\nassert all(_recipe_only(m) == _recipe_only(_metas[0]) for m in _metas), \'Cache shards come from different recipes\'\nCACHE_META = _recipe_only(_metas[0])\n_shards = sorted(m.get(\'shard\', 0) for m in _metas)\nassert _shards == list(range(_metas[0].get(\'n_shards\', 1))), f\'Attach every cache shard exactly once (found {_shards})\'\nassert CACHE_META.get(\'version\') in (\'knee-2c-v2\', \'knee-2c-v3\', \'knee-2c-v4\', \'knee-2c-v5\'), \'Rebuild the step-2c cache with current preprocessing\'\nPLANES = CACHE_META[\'slots\']\nCACHE_IMG = CACHE_META[\'img\']\nN_SLICES = CACHE_META[\'n_slices\']\n\n# ---- model recipe -------------------------------------------------------------\n# Eight self-trained families, chosen by PRESET (one launcher set per family):\n# reader_convnext: public 2.5D study reader (0.929 alone). ConvNeXt-T on windows of three\n#   adjacent slices, slot and stack-position tokens, two transformer layers, attention\n#   pooling per finding, EMA weights, warmup then cosine.\n# reader_student: the ConvNeXt reader trained on report labels mixed with our teachers\' OOF predictions.\n# reader_student2: round 2, whose teacher also includes the round-1 students.\n# reader_caformer: the same reader on CAFormer-S18 (ImageNet-22k), for backbone diversity.\n# reader_coatnet: the same reader on CoAtNet-RMLP-2 at 384 px, the backbone of the public\n#   Raptor single model (0.928 alone). Gradient checkpointing keeps it inside a T4.\n# slot_dinov2: the public DINOv2 member (one spread 3-slice image per slot, last six\n#   blocks fine-tuned, per-finding query head).\n# radimagenet: frozen official RadImageNet ResNet-50 slice tokens with a per-finding query\n#   head (public E9/E11 arm); encoded once, all folds in one run (rad_runtime.py).\n# kneenet_b0: the earlier EfficientNet-b0 model, kept for comparison.\nPRESET = os.environ.get("KNEE_PRESET", "reader_convnext")\nPRESETS = {\n    \'reader_convnext\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                            epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                            clip=3., grad_checkpoint=False),\n    # Noisy student: the same ConvNeXt reader recipe trained on targets mixed with the out-of-fold\n    # predictions of our own readers (teacher_targets.csv). Agreement-gated: the teacher share runs\n    # from 0.5 where the report readers agree confidently to 0.9 where they are split or silent.\n    # Slot dropout hides whole MRI sequences in training (test sites differ in protocol).\n    \'reader_student\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                           epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                           pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                           clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # Round 2: the same student recipe, whose teacher also averages the round-1 students\' OOF.\n    # 11 epochs: pseudo-validation plateaus by epochs 10-12, and a two-fold session then fits\n    # in under two GPU hours.\n    \'reader_student2\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                            epochs=11, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                            pct_start=1/11, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                            clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,\n                            teacher_families={\'reader_convnext\': 1.0, \'reader_caformer\': 1.0, \'reader_student\': 1.0}),\n    # Same reader, a different backbone family for ensemble diversity (convolution stages, then\n    # attention; ImageNet-22k). ~23% more activation memory per image than ConvNeXt-T, whose\n    # batch 4 peaked at 10.9 GB on a T4: half the micro-batch, same effective batch.\n    \'reader_caformer\': dict(arch=\'reader25d\', backbone=\'caformer_s18.sail_in22k_ft_in1k\', img=CACHE_IMG,\n                            epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                            clip=3., grad_checkpoint=False),\n    # Round-1 student recipe and teacher on a ResNet-50 (batch-norm CNN) for ensemble diversity:\n    # LB 2026-10-09/10: more ConvNeXt students/seeds added ~0.001; round-2 teachers did not help.\n    # Half the micro-batch (activation memory), same effective batch.\n    \'reader_student_r50\': dict(arch=\'reader25d\', backbone=\'resnet50.a1_in1k\', img=CACHE_IMG,\n                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # Report-grounded student: the round-1 student recipe and teacher plus an auxiliary loss that\n    # regresses each study\'s report embedding (scripts/report_embeddings.py; attach the\n    # knee-report-embeddings dataset) from the images, at this weight relative to the label BCE.\n    \'reader_student_rep\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                               epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,\n                               report_aux=.2),\n    # Round-1 student recipe and teacher on CAFormer-S18 (in22k): its readers (session J) beat the\n    # ConvNeXt readers on pseudo-validation (AUC 0.895/0.897 vs 0.893/0.891, folds 0/1); ResNet-50\n    # students fell far behind (0.915 vs 0.963), so diversity comes from this family instead.\n    \'reader_student_caf\': dict(arch=\'reader25d\', backbone=\'caformer_s18.sail_in22k_ft_in1k\', img=CACHE_IMG,\n                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # Round-1 student recipe and teacher on ConvNeXt-Base (ImageNet-22k), ~3x ConvNeXt-T compute:\n    # trained on the TPU (08_tpu_student_cnxb), where the long run does not hold up GPU sessions.\n    \'reader_student_cnxb\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # The round-1 ConvNeXt student recipe and teacher on the knee-2c-v4 cache (six v3 slots + two\n    # 90 mm local slots around the header-geometry joint anchor); a separate family name so its\n    # LB comparison with the v3 students (0.935 on folds 0-1) stays clean.\n    \'reader_student_v4\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                              epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # ConvNeXt-Base student on knee-2c-v4 (TPU): LB 2026-10-12 showed both gains separately,\n    # ConvNeXt-B v3 x5 = 0.938 vs ConvNeXt-T v3 x6 = 0.936, and v4 folds 0-1 = 0.938 vs v3 = 0.935.\n    \'reader_student_cnxb_v4\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                   epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                   pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                   clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # The ConvNeXt student on knee-2c-v5 (288 px, 10 windows per series; TPU).\n    # 288 px x 8 slots: half the T4 micro-batch of v4 (activation memory), same effective batch of 8.\n    \'reader_student_v5\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                              epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # ConvNeXt-Base and CAFormer-S18 students on knee-2c-v5 (TPU / GPU after the 2026-10-17 quota reset).\n    # On a T4: one study per micro-batch (48 windows at 288 px) with gradient checkpointing; the TPU\n    # runtime ignores both (8 studies per step across the chips, no checkpointing). 8 epochs: at 50-60\n    # min per epoch on a T4, 14 would exceed the 12 h session (2026-10-14); ConvNeXt-B at 288 px\n    # reached val BCE 0.375 / AUC 0.954 by epoch 3.\n    \'reader_student_cnxb_v5\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                   epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                   pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                   clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2),\n    # v5 students taught partly by heliosli\'s compact model (its out-of-fold predictions on the training\n    # studies, 10_compact_oof_*): teacher = our ConvNeXt readers (sessions A-F, the Z5/B5 teacher; no CAFormer\n    # reader outputs are attached) and the compact model at 1.0 each, i.e. half ours and half compact, its\n    # per-finding ranking put on our teachers\' value distribution (its report targets use other label\n    # definitions, e.g. synovitis mean 0.54 vs 0.30);\n    # otherwise the ConvNeXt-T / -B v5 recipes.\n    \'reader_student_ct_v5\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                                 epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                 pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                 clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,\n                                 teacher_families={\'reader_convnext\': 1.0, \'reader_compact\': 1.0},\n                                 teacher_quantile_map=(\'reader_compact\',)),\n    \'reader_student_cnxb_ct_v5\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                      epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                      pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                      clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2,\n                                      teacher_families={\'reader_convnext\': 1.0, \'reader_compact\': 1.0},\n                                      teacher_quantile_map=(\'reader_compact\',)),\n    \'reader_student_caf_v5\': dict(arch=\'reader25d\', backbone=\'caformer_s18.sail_in22k_ft_in1k\', img=CACHE_IMG,\n                                  epochs=7, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                  pct_start=1/7, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                  clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    \'reader_coatnet\': dict(arch=\'reader25d\', backbone=\'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\', img=384,\n                           epochs=10, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                           pct_start=1/10, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                           clip=3., grad_checkpoint=True),\n    \'slot_dinov2\': dict(arch=\'slot_query\', backbone=\'vit_small_patch14_dinov2.lvd142m\', img=224, epochs=10,\n                        batch=4, accum=2, lr=1e-3, encoder_lr_scale=8e-3, wd=.02, pct_start=.15, dropout=.2,\n                        unfreeze_last=6, head_hidden=256, ema=0., clip=5., grad_checkpoint=False),\n    \'radimagenet\': dict(arch=\'radhead\', backbone=\'radimagenet_resnet50\', img=224, epochs=24, batch=48,\n                        accum=1, lr=2e-4, encoder_lr_scale=0., wd=3e-3, pct_start=0., dropout=.15,\n                        unfreeze_last=None, head_hidden=512, ema=0., clip=1., grad_checkpoint=False),\n    \'kneenet_b0\': dict(arch=\'kneenet\', backbone=\'tf_efficientnet_b0.ns_jft_in1k\', img=CACHE_IMG, epochs=6,\n                       batch=4, accum=1, lr=3e-4, encoder_lr_scale=.3, wd=1e-2, pct_start=.1, dropout=.4,\n                       unfreeze_last=None, head_hidden=None, ema=0., clip=5., grad_checkpoint=False),\n}\n_preset = PRESETS[PRESET]\n_needs = {\'_v4\': \'knee-2c-v4\', \'_v5\': \'knee-2c-v5\'}.get(PRESET[-3:])\nassert _needs is None or CACHE_META.get(\'version\') == _needs, f\'Attach the {_needs} cache outputs\'\nARCH = _preset[\'arch\']\nBACKBONE = os.environ.get("KNEE_BACKBONE", _preset[\'backbone\'])\nIMG = _preset[\'img\']              # model input size; the cache is resized on the GPU if different\nEPOCHS = _preset[\'epochs\']\nBATCH = _preset[\'batch\']          # studies per micro-batch\nACCUM_STEPS = _preset[\'accum\']    # effective batch = BATCH * ACCUM_STEPS studies\nLR = _preset[\'lr\']                # head learning rate\nENCODER_LR_SCALE = _preset[\'encoder_lr_scale\']   # encoder rate = LR * this\nWD = _preset[\'wd\']\nPCT_START = _preset[\'pct_start\']  # warmup share of the one-cycle schedule\nDROPOUT = _preset[\'dropout\']\nUNFREEZE_LAST = _preset[\'unfreeze_last\']\nHEAD_HIDDEN = _preset[\'head_hidden\']\nEMA_DECAY = _preset[\'ema\']        # 0 disables; EMA weights are validated and saved\nCLIP = _preset[\'clip\']\nGRAD_CHECKPOINT = _preset[\'grad_checkpoint\']\nTEACHER_WEIGHT = _preset.get(\'teacher\', 0.0)   # constant or (lo, hi) gate; attach teacher_targets.csv\nSLOT_DROPOUT = _preset.get(\'slot_dropout\', 0.0) # reader25d: chance of hiding each present series in training\nTEACHER_FAMILY_WEIGHTS = _preset.get(\'teacher_families\')   # None: label_builder.TEACHER_FAMILIES\nTEACHER_QUANTILE_MAP = _preset.get(\'teacher_quantile_map\', ())   # outside teachers put on our teachers\' scale\nREPORT_AUX = _preset.get(\'report_aux\', 0.0)    # weight of the report-embedding auxiliary loss; 0 disables\nTRAIN_WINDOWS = 6                 # reader25d: windows encoded per series per step (of 8)\n# Gold studies train in every fold at this per-cell weight (decision 2026-10-07). Their\n# audit AUC is then in-sample and is printed as such; selection never reads gold.\nGOLD_WEIGHT = 3.0\nREADER_WEIGHTS = {}               # e.g. {\'extra_labels_sol_v2.csv\': 2.0}; default 1 each\n# Every reader the run must find; a missing attachment stops before any GPU time is spent.\nREQUIRED_READERS = [\'report_labels_v2.csv\', \'llm_labels_v4_blend.csv\', \'labels_llm_gpt56sol.csv\',\n                    \'qwen_knee_weak_labels.csv\', \'extra_labels_sol_v2.csv\',\n                    \'extra_labels_sonnet_legacy.csv\', \'pseudo_google_gemini*flash*.jsonl\']\nNORMALIZATION = \'pretrained\'\nN_FOLDS = 5\nFOLDS_TO_RUN = json.loads(os.environ.get("KNEE_FOLDS", "[0,1,2,3,4]"))  # one fold first on Kaggle\nGROUP_COL = \'report_group\'        # folds keep duplicate reports together\nRESUME = False  # copy both fold_N.pt and last_fold_N.pt to OUT before enabling\nT_TRAIN     = 10       # kneenet only: contiguous slices sampled per plane per step\nSEED        = int(os.environ.get("KNEE_SEED", 42))   # sessions may train extra seeds of a family\nNUM_WORKERS = int(os.environ.get("KNEE_WORKERS", 4))   # Kaggle gives 4 cores; two-GPU sessions use 2 per job\n\ndef write_atomic(path, content):\n    # Two jobs in one session may write the same small file (folds, config); replace it whole.\n    # content: text, or a function that writes to the temporary path it is given.\n    temp = f\'{path}.{os.getpid()}.tmp\'\n    if callable(content):\n        content(temp)\n    else:\n        with open(temp, \'w\') as handle: handle.write(content)\n    os.replace(temp, path)\n\n\nos.makedirs(OUT, exist_ok=True)\nDEV = "cuda" if torch.cuda.is_available() else "cpu"\ntorch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)\ntorch.backends.cudnn.benchmark = True          # shapes are fixed -> let cuDNN autotune\n\nprint("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")\nif DEV == "cuda" and torch.cuda.device_count() > 1:\n    print(f"note: {torch.cuda.device_count()} GPUs visible, using 1 by design "\n          f"(DataParallel benchmarked 13x slower here)")\nif ARCH == \'kneenet\' and BACKBONE.startswith(\'vit_\'):\n    BATCH, ACCUM_STEPS = 1, 4  # effective batch 4; lower activation memory\nprint(f"recipe: {PRESET} ({ARCH}) | {BACKBONE} @ {IMG}px | {CACHE_META[\'version\']} {len(PLANES)} slots x "\n      f"{N_SLICES} slices x {CACHE_IMG}px in {len(CACHE_DIRS)} shard(s) | {EPOCHS} epochs, "\n      f"{BATCH}x{ACCUM_STEPS} studies/step | gold weight {GOLD_WEIGHT}")\nassert DEV == "cuda" or os.environ.get("KNEE_ALLOW_CPU"), "Enable a GPU before starting training"\n\nRESUME_DIR = os.environ.get("KNEE_RESUME_DIR", "")\nif RESUME_DIR:\n    import shutil\n    for fold in FOLDS_TO_RUN:\n        for filename in (f\'fold_{fold}.pt\', f\'last_fold_{fold}.pt\'):\n            previous = Path(RESUME_DIR) / os.environ.get(\'KNEE_RUN_NAME\', PRESET) / filename\n            if not previous.is_file():\n                continue                       # this fold has no earlier progress\n            destination = Path(OUT) / filename\n            assert not destination.exists(), f\'Restart the session before copying {destination}\'\n            shutil.copy2(previous, destination)\n            RESUME = True\n\n\n"""Soft training targets from report-label readers; embedded in training.ipynb."""\nimport os, re, json, fnmatch, hashlib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\n# Public report-label tables (rules allow public external data). File name -> suffix of\n# the twelve target columns. The mean of these four reached 0.896 macro AUC on the gold\n# studies in the public 2.5D reader\'s own measurement.\nPUBLIC_LABEL_TABLES = {\n    \'report_labels_v2.csv\': \'\',            # pilkwang/rsna-knee-llm-labels\n    \'llm_labels_v4_blend.csv\': \'\',         # stevenleehans/rsna-knee-llm-report-labels\n    \'labels_llm_gpt56sol.csv\': \'\',         # lixin73/rsna-knee-llm-report-labels-sol56\n    \'qwen_knee_weak_labels.csv\': \'__label\',  # laymond/rsna-knee-abnormality-qwen3-8b-weak-labels\n}\n# Our own readers:\n#  - extra_labels_*.csv from scripts/export_reader_labels.py (GPT Sol v2, Sonnet legacy);\n#  - the Flash member file written by our 01_labels notebook (its labels/ folder), read\n#    directly so Flash counts once. That notebook\'s consensus train_targets.csv is not read:\n#    it already mixes Flash with Sonnet.\nOWN_PREFIX = \'extra_labels\'\nOWN_JSONL_READERS = {\'pseudo_google_gemini*flash*.jsonl\': \'legacy\'}\n\n# State -> probability that the finding is positive. Legacy is the 01_labels notebook\'s own\n# HARD map; v2 states follow the knee-report-labels-v2 protocol. Unsure abstains (NaN)\n# under legacy; under v2 it marks an affirmative finding of undecidable eligibility.\nSTATE_P = {\n    \'legacy\': {\'definite\': 1.0, \'probable\': .75, \'possible\': .35, \'absent\': 0.0},\n    \'v2\': {\'definite\': 1.0, \'unsure\': .5, \'absent\': 0.0},\n}\nSTATE_KEYS = dict(zip(\n    ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n     "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"],\n    ["acl", "mcl", "medial_meniscus", "lateral_meniscus", "medial_oa", "lateral_oa",\n     "pf_oa", "effusion", "synovitis", "bakers", "contusion", "fracture"]))\n\n\ndef iter_jsonl(path):\n    # One historical shard was written with literal "\\n" separators; split on both.\n    with open(path) as handle:\n        for chunk in re.split(r\'\\n|\\\\n\', handle.read()):\n            if chunk.strip():\n                yield json.loads(chunk)\n\n\ndef states_frame(records, label_cols, protocol):\n    mapping = STATE_P[protocol]\n    rows = [{\'StudyInstanceUID\': str(r[\'StudyInstanceUID\']),\n             **{c: mapping.get(str(r.get(STATE_KEYS.get(c, c), \'\')).lower()) for c in label_cols}}\n            for r in records]\n    frame = pd.DataFrame(rows, columns=[\'StudyInstanceUID\', *label_cols])\n    return frame.drop_duplicates(\'StudyInstanceUID\', keep=\'last\')\n\n\ndef find_label_tables(root=\'/kaggle/input\'):\n    found = {}\n    for path, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in (\'train_series\', \'test_series\')]\n        for name in files:\n            if (name in PUBLIC_LABEL_TABLES or (name.startswith(OWN_PREFIX) and name.endswith(\'.csv\'))\n                    or any(fnmatch.fnmatch(name, pattern) for pattern in OWN_JSONL_READERS)):\n                found.setdefault(name, []).append(os.path.join(path, name))\n    duplicates = {k: v for k, v in found.items() if len(v) > 1}\n    assert not duplicates, f\'Attach each label table once: {duplicates}\'\n    flash = [k for k in found if any(fnmatch.fnmatch(k, p) for p in OWN_JSONL_READERS)]\n    assert len(flash) <= 1, f\'More than one Flash member file attached: {flash}\'\n    return {k: v[0] for k, v in sorted(found.items())}\n\n\ndef read_reader(path, label_cols):\n    name = os.path.basename(path)\n    if name.endswith(\'.jsonl\'):\n        protocol = next(v for p, v in OWN_JSONL_READERS.items() if fnmatch.fnmatch(name, p))\n        frame = states_frame(iter_jsonl(path), label_cols, protocol)\n        columns = list(label_cols)\n    else:\n        suffix = PUBLIC_LABEL_TABLES.get(name, \'\')\n        frame = pd.read_csv(path, dtype={\'StudyInstanceUID\': str})\n        columns = [c + suffix for c in label_cols]\n    missing = [c for c in columns if c not in frame.columns]\n    assert not missing, f\'{name} lacks {missing[:3]}\'\n    assert not frame[\'StudyInstanceUID\'].duplicated().any(), f\'{name} repeats studies\'\n    values = frame.set_index(\'StudyInstanceUID\')[columns].apply(pd.to_numeric, errors=\'coerce\')\n    values.columns = label_cols\n    bad = (values < 0) | (values > 1)\n    if bad.any().any():\n        print(f\'{name}: {int(bad.sum().sum())} values outside 0..1 treated as unobserved\')\n        values = values.mask(bad)\n    return values\n\n\n# Noisy-student targets (2026-10-08). teacher_targets.csv holds out-of-fold predictions of our own\n# image models (scripts/teacher_targets.py); image models average out report noise, so their OOF\n# predictions are cleaner targets than any single report reader. Gold rows never take teacher values.\nTEACHER_FILE = \'teacher_targets.csv\'\n\n\ndef find_teacher(root=\'/kaggle/input\'):\n    hits = []\n    for path, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in (\'train_series\', \'test_series\')]\n        if TEACHER_FILE in files:\n            hits.append(os.path.join(path, TEACHER_FILE))\n    assert len(hits) <= 1, f\'Attach one teacher table, found {hits}\'\n    return hits[0] if hits else None\n\n\n# Family -> weight for teachers. Readers only: on the public LB the slot and RadImageNet families\n# lowered the blend (17 models at equal weights 0.917 < two reader folds 0.923). Students and\n# third-party checkpoints are never teachers.\nTEACHER_FAMILIES = {\'reader_convnext\': 1.0, \'reader_caformer\': 1.0}\n\n\ndef build_teacher(root, label_cols, gold_ids=(), families=None, quantile_map=()):\n    """Teacher probabilities from the out-of-fold predictions (oof_fold_N.csv) of our own models\n    found under root: seeds of a family are averaged, then families by weight. Every value is\n    out-of-fold. Families in quantile_map (an outside model trained on other label definitions)\n    keep their per-finding ranking but take the value distribution of the weighted mean of the\n    other families before the average. Returns (frame with StudyInstanceUID, label_cols, n_members; meta)."""\n    families = TEACHER_FAMILIES if families is None else families\n    family_of = lambda folder: re.sub(r\'_s\\d+$\', \'\', folder)\n    members = {}\n    for path, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in (\'train_series\', \'test_series\')]\n        folder = os.path.basename(path)\n        oofs = sorted(f for f in files if re.fullmatch(r\'oof_fold_\\d+\\.csv\', f))\n        if oofs and family_of(folder) in families:\n            members.setdefault(folder, []).extend(\n                pd.read_csv(os.path.join(path, f), dtype={\'StudyInstanceUID\': str}) for f in oofs)\n    assert members, f\'no teacher OOF predictions under {root}\'\n    gold_ids, per_family = set(gold_ids), {}\n    meta = dict(members={}, families={}, weights=dict(families))\n    for folder, parts in sorted(members.items()):\n        frame = pd.concat(parts, ignore_index=True)\n        assert frame[\'StudyInstanceUID\'].is_unique, f\'{folder}: a study is out-of-fold twice\'\n        assert not gold_ids & set(frame[\'StudyInstanceUID\']), f\'{folder}: gold study in OOF\'\n        meta[\'members\'][folder] = dict(studies=len(frame), folds=sorted(int(f) for f in frame[\'fold\'].unique()))\n        values = frame.set_index(\'StudyInstanceUID\')[label_cols].astype(float)\n        assert ((values >= 0) & (values <= 1)).all().all(), f\'{folder}: OOF outside 0..1\'\n        per_family.setdefault(family_of(folder), []).append(values)\n    ids = sorted(set().union(*[f.index for frames in per_family.values() for f in frames]))\n    num = np.zeros((len(ids), len(label_cols))); den = np.zeros_like(num)\n    n_members = np.zeros(len(ids), int)\n    means = {}\n    for family, frames in per_family.items():\n        stack = np.stack([f.reindex(ids).to_numpy(float) for f in frames])\n        counts = np.isfinite(stack).sum(axis=0)\n        means[family] = (np.where(counts > 0, np.nansum(stack, axis=0) / np.maximum(counts, 1), np.nan), stack)\n    mapped = [f for f in quantile_map if f in means]\n    if mapped:\n        ref_num = sum(np.nan_to_num(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)\n        ref_den = sum(np.isfinite(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)\n        assert np.max(ref_den) > 0, \'quantile_map needs at least one other teacher family\'\n        ref = np.where(ref_den > 0, ref_num / np.maximum(ref_den, 1e-9), np.nan)\n        for family in mapped:\n            m = means[family][0].copy()\n            for j in range(m.shape[1]):\n                ok, r = np.isfinite(m[:, j]), np.isfinite(ref[:, j])\n                if ok.any() and r.any():\n                    pct = (pd.Series(m[ok, j]).rank(method=\'average\').to_numpy() - .5) / ok.sum()\n                    m[ok, j] = np.quantile(ref[r, j], pct)\n            means[family] = (m, means[family][1])\n        meta[\'quantile_mapped\'] = mapped\n    for family, (mean, stack) in means.items():\n        ok = np.isfinite(mean)\n        num += np.where(ok, mean, 0) * families[family]\n        den += ok * families[family]\n        n_members += np.isfinite(stack).all(axis=2).sum(axis=0)\n        meta[\'families\'][family] = dict(members=len(frames), studies=int(ok.all(axis=1).sum()))\n    teacher = pd.DataFrame(np.where(den > 0, num / np.maximum(den, 1e-9), np.nan), columns=label_cols)\n    teacher.insert(0, \'StudyInstanceUID\', ids)\n    teacher[\'n_members\'] = n_members\n    return teacher, meta\n\n\ndef apply_teacher(frame, teacher_path, label_cols, weight):\n    """Mix teacher probabilities into the pseudo rows: y = (1 - a) * report + a * teacher.\n    weight is a constant a, or a gate (lo, hi): a = lo + (hi - lo) * u with u = 1 - |2 * report - 1|,\n    so cells where the report readers agree confidently (consensus near 0 or 1) keep more of the\n    report and split or uncertain cells lean on the image teacher. A pseudo cell without a report\n    label takes the teacher value at cell weight hi. Gold rows are returned unchanged.\n    Returns (frame, summary)."""\n    lo, hi = (weight, weight) if np.isscalar(weight) else tuple(weight)\n    assert 0 <= lo <= hi <= 1 and hi > 0\n    teacher = pd.read_csv(teacher_path, dtype={\'StudyInstanceUID\': str})\n    assert teacher[\'StudyInstanceUID\'].is_unique, \'teacher repeats studies\'\n    t = teacher.set_index(\'StudyInstanceUID\')[label_cols].apply(pd.to_numeric, errors=\'coerce\')\n    assert (t.isna() | ((t >= 0) & (t <= 1))).all().all(), \'teacher values outside 0..1\'\n    out = frame.copy()\n    rows = np.flatnonzero(out[\'source\'].eq(\'pseudo\').to_numpy())\n    wcols = [f\'w_{c}\' for c in label_cols]\n    y = np.array(out.loc[out.index[rows], label_cols], dtype=float)\n    w = np.array(out.loc[out.index[rows], wcols], dtype=float)\n    tv = np.array(t.reindex(out[\'StudyInstanceUID\'].iloc[rows]), dtype=float)\n    has_t = np.isfinite(tv)\n    both, only_t = has_t & np.isfinite(y), has_t & ~np.isfinite(y)\n    a = lo + (hi - lo) * (1 - np.abs(2 * np.where(np.isfinite(y), y, .5) - 1))\n    y[both] = (1 - a[both]) * y[both] + a[both] * tv[both]\n    y[only_t] = tv[only_t]\n    w[only_t] = hi\n    out.loc[out.index[rows], label_cols] = np.where(np.isfinite(y), np.clip(y, .01, .99), np.nan).astype(np.float32)\n    out.loc[out.index[rows], wcols] = w.astype(np.float32)\n    summary = dict(teacher=teacher_path, weight=[lo, hi], mean_teacher_share=float(a[both].mean()) if both.any() else 0.,\n                   studies=int(has_t.any(axis=1).sum()),\n                   pseudo_studies=len(rows), mixed_cells=int(both.sum()), filled_cells=int(only_t.sum()),\n                   sha256=hashlib.sha256(open(teacher_path, \'rb\').read()).hexdigest())\n    return out, summary\n\n\ndef report_groups(reports):\n    # Byte-identical (up to case/punctuation) reports share targets: keep them in one fold.\n    norm = reports.fillna(\'\').astype(str).str.lower().map(lambda t: re.sub(r\'\\W+\', \'\', t))\n    return norm.map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:16]).to_numpy()\n\n\ndef build_targets(train, label_cols, tables, reader_weights=None, gold_weight=3.0):\n    """train: competition train.csv. Returns (frame, report) where frame holds soft\n    targets, per-cell weights (w_<col>), source and report group per study."""\n    ids = train[\'StudyInstanceUID\'].astype(str)\n    readers = {name: read_reader(path, label_cols).reindex(ids) for name, path in tables.items()}\n    assert readers, \'Attach at least one report-label table\'\n    weights = {name: (reader_weights or {}).get(name, 1.0) for name in readers}\n    num = sum(r[label_cols].fillna(0).to_numpy() * weights[n] for n, r in readers.items())\n    den = sum(r[label_cols].notna().to_numpy() * weights[n] for n, r in readers.items())\n    with np.errstate(invalid=\'ignore\', divide=\'ignore\'):\n        soft = np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)\n    soft = np.clip(soft, .01, .99)\n    gold = train[label_cols].apply(pd.to_numeric, errors=\'coerce\')\n    is_gold = gold.notna().all(axis=1).to_numpy()\n    y = soft.astype(np.float32)\n    w = np.where(np.isfinite(soft), 1.0, 0.0).astype(np.float32)\n    y[is_gold] = gold.to_numpy(np.float32)[is_gold]\n    w[is_gold] = gold_weight\n    frame = pd.DataFrame(y, columns=label_cols)\n    frame.insert(0, \'StudyInstanceUID\', ids.to_numpy())\n    for k, c in enumerate(label_cols): frame[f\'w_{c}\'] = w[:, k]\n    frame[\'source\'] = np.where(is_gold, \'gold\', \'pseudo\')\n    frame[\'report_group\'] = report_groups(train[\'Report\']) if \'Report\' in train else ids.to_numpy()\n    # Diagnostic only: each reader against the image-read gold labels. Not used for any choice.\n    report = {}\n    for name, r in {**readers, \'consensus\': pd.DataFrame(soft, index=ids, columns=label_cols)}.items():\n        aucs = []\n        # A reader that never labelled the gold studies has no gold AUC (not 0.5).\n        gold_rows = int(r.loc[ids[is_gold], label_cols].notna().any(axis=1).sum())\n        for c in label_cols:\n            truth, score = gold.loc[is_gold, c].to_numpy(), r.loc[ids[is_gold], c].fillna(.5).to_numpy()\n            aucs.append(roc_auc_score(truth, score) if len(np.unique(truth)) == 2 and gold_rows else np.nan)\n        finite = [a for a in aucs if np.isfinite(a)]\n        report[name] = dict(gold_macro_auc=float(np.mean(finite)) if finite else float(\'nan\'),\n                            gold_rows=gold_rows, weight=weights.get(name, 0),\n                            coverage=float(np.isfinite(r.to_numpy()).mean()))\n    return frame, report\n\n# ---- cache shards -------------------------------------------------------------\n# Rows are addressed as (shard, row in shard); every shard shares one recipe.\nparts = []\nfor shard_id, directory in enumerate(CACHE_DIRS):\n    part = pd.read_csv(os.path.join(directory, "cache_index.csv"))\n    shape = np.load(os.path.join(directory, "cache.npy"), mmap_mode="r").shape\n    assert shape[1:] == (len(PLANES), N_SLICES, CACHE_IMG, CACHE_IMG), f\'{directory}: cache shape {shape}\'\n    assert len(part) == shape[0] and part[\'row\'].between(0, shape[0] - 1).all()\n    parts.append(part.assign(cache_id=shard_id))\n    print(f"cache shard {shard_id}: {shape[0]:,} studies, {np.prod(shape)/1e9:.1f} GB  ({directory})")\nindex = pd.concat(parts, ignore_index=True)\nassert index[\'StudyInstanceUID\'].is_unique, \'a study appears in two cache shards\'\nT_TRAIN = min(T_TRAIN, N_SLICES)\n\n# ---- targets: report-label readers + gold ------------------------------------\ntrain_csv = pd.read_csv(os.path.join(COMP_DIR, \'train.csv\'), dtype={\'StudyInstanceUID\': str})\nLABEL_TABLES = find_label_tables(LABEL_ROOT)\nprint(\'label readers:\', list(LABEL_TABLES))\n_missing = [r for r in REQUIRED_READERS if not any(fnmatch.fnmatch(name, r) for name in LABEL_TABLES)]\nassert not _missing, f\'Attach the label inputs providing {_missing} (see KAGGLE_RUN/README.md)\'\ntargets, label_report = build_targets(train_csv, LABEL_COLS, LABEL_TABLES, READER_WEIGHTS, GOLD_WEIGHT)\nprint(pd.DataFrame(label_report).T.round(3).to_string())\nprint(\'(gold AUC of each reader is a diagnostic only; it chooses nothing. NaN: the reader has no gold rows)\')\nTEACHER_SUMMARY = None\nif np.max(TEACHER_WEIGHT) > 0:\n    _teacher = find_teacher(LABEL_ROOT)\n    if _teacher is None:\n        # No teacher table attached: build it here from the attached reader session outputs\n        # (their out-of-fold predictions), exactly as 07_teacher_targets would.\n        _gold = set(targets.loc[targets[\'source\'].eq(\'gold\'), \'StudyInstanceUID\'])\n        _frame, _meta = build_teacher(LABEL_ROOT, LABEL_COLS, gold_ids=_gold, families=globals().get(\'TEACHER_FAMILY_WEIGHTS\'),\n                                      quantile_map=globals().get(\'TEACHER_QUANTILE_MAP\', ()))\n        print(\'teacher built in session from:\', json.dumps(_meta[\'members\']))\n        _absent = sorted(f for f, w in (globals().get(\'TEACHER_FAMILY_WEIGHTS\') or {}).items() if w > 0 and f not in _meta[\'families\'])\n        assert not _absent, f\'Attach the teacher inputs for families {_absent}\'\n        os.makedirs(OUT, exist_ok=True)\n        _teacher = os.path.join(OUT, \'teacher_targets_used.csv\')\n        write_atomic(_teacher, lambda p: _frame.to_csv(p, index=False))\n    targets, TEACHER_SUMMARY = apply_teacher(targets, _teacher, LABEL_COLS, TEACHER_WEIGHT)\n    print(\'teacher targets:\', TEACHER_SUMMARY)\nwrite_atomic(os.path.join(OUT, \'targets_used.csv\'), lambda p: targets.to_csv(p, index=False))\n\nWEIGHT_COLS = [f\'w_{c}\' for c in LABEL_COLS]\ndf = index[index["n_present"] > 0].merge(targets, on="StudyInstanceUID", how="inner")\ndf = df[df[LABEL_COLS].notna().any(axis=1)].reset_index(drop=True)\nprint(f"\\nusable studies: {len(df):,}  ({df[\'source\'].value_counts().to_dict()})")\n# Report-embedding auxiliary targets (training only): NaN rows have no embedding and are masked.\nREP_COLS, AUX_DIM, REPORT_AUX_SUMMARY = [], 0, None\nif globals().get(\'REPORT_AUX\', 0.0) > 0:\n    _rep_path = one_input(\'report_embeddings.csv\', root=LABEL_ROOT)\n    _rep = pd.read_csv(_rep_path, dtype={\'StudyInstanceUID\': str})\n    REP_COLS = [c for c in _rep.columns if c.startswith(\'rep_\')]\n    assert REP_COLS and _rep[\'StudyInstanceUID\'].is_unique and not set(REP_COLS) & set(df.columns)\n    df = df.merge(_rep[[\'StudyInstanceUID\', *REP_COLS]], on=\'StudyInstanceUID\', how=\'left\')\n    AUX_DIM = len(REP_COLS)\n    import hashlib as _hashlib\n    REPORT_AUX_SUMMARY = dict(weight=REPORT_AUX, dims=AUX_DIM,\n                              studies=int(df[REP_COLS[0]].notna().sum()),\n                              sha256=_hashlib.sha256(open(_rep_path, \'rb\').read()).hexdigest())\n    print(\'report auxiliary targets:\', REPORT_AUX_SUMMARY)\ny_all = df[LABEL_COLS].to_numpy(np.float32)\nprint(f"observed label cells: {(~np.isnan(y_all)).mean():.1%}")\nprint("\\nmean soft target (observed cells):")\nprint(pd.Series(np.nanmean(y_all[df[\'source\'].eq(\'pseudo\')], axis=0), index=LABEL_COLS).round(3).to_string())\nassert df[\'source\'].isin([\'gold\',\'pseudo\']).all(), \'Unknown label source\'\nassert ((np.isnan(y_all)) | ((y_all>=0)&(y_all<=1))).all(), \'Targets outside [0,1]\'\ngold_check = train_csv.set_index(\'StudyInstanceUID\')[LABEL_COLS].apply(pd.to_numeric, errors=\'coerce\')\ngold_ids = df.loc[df[\'source\'].eq(\'gold\'), \'StudyInstanceUID\']\nassert np.array_equal(df.loc[df[\'source\'].eq(\'gold\'), LABEL_COLS].to_numpy(float),\n                      gold_check.loc[gold_ids].to_numpy(float)), \'Gold rows must carry the official labels\'\n\n"""Fold creation and metrics with explicit gold exclusion."""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\n\ndef assign_folds(df, label_cols, n_splits=5, seed=42, group_col=None):\n    folds = np.full(len(df), -1, dtype=int)\n    pseudo = np.flatnonzero(df[\'source\'].eq(\'pseudo\').to_numpy())\n    assert len(pseudo) >= n_splits\n    y = df.iloc[pseudo][label_cols].to_numpy(float)\n    # Balance missingness as well as positive findings. No gold target enters the split.\n    strata = np.concatenate([np.nan_to_num(y, nan=0) >= .5, np.isfinite(y)], axis=1).astype(int)\n    groups = df.iloc[pseudo][group_col].astype(str).to_numpy() if group_col else df.iloc[pseudo][\'StudyInstanceUID\'].to_numpy()\n    if len(np.unique(groups)) < len(groups):\n        splitter = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        splits = splitter.split(pseudo, groups=groups)\n    else:\n        splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        splits = splitter.split(pseudo, strata)\n    for fold, (_, val) in enumerate(splits): folds[pseudo[val]] = fold\n    assert (folds[pseudo] >= 0).all()\n    assert (folds[df[\'source\'].eq(\'gold\')] == -1).all()\n    return folds\n\n\ndef macro_auc(y, scores, label_cols):\n    aucs = {}\n    for k, name in enumerate(label_cols):\n        ok = np.isfinite(y[:, k]) & np.isfinite(scores[:, k])\n        binary = (y[ok, k] >= .5).astype(int)\n        aucs[name] = roc_auc_score(binary, scores[ok, k]) if len(np.unique(binary)) == 2 else np.nan\n    valid = [v for v in aucs.values() if np.isfinite(v)]\n    return (float(np.mean(valid)) if valid else float(\'nan\')), aucs\n\n\ndef rank_blend(predictions):\n    arrays = [np.asarray(p, dtype=float) for p in predictions]\n    assert arrays and all(a.shape == arrays[0].shape for a in arrays)\n    assert all(np.isfinite(a).all() for a in arrays)\n    return np.mean([pd.DataFrame(a).rank(pct=True, method=\'average\').to_numpy() for a in arrays], axis=0)\n\n"""Training dataset, embedded in training.ipynb after the fold helpers."""\nimport cv2\n\nclass KneeDS(Dataset):\n    def __init__(self, df, rows, train):\n        self.df = df.reset_index(drop=True)\n        self.rows = rows\n        self.train = train\n        self.y = self.df[LABEL_COLS].to_numpy(np.float32)\n        # Per-cell loss weights (gold rows are up-weighted); unweighted tables default to 1.\n        weight_cols = [f\'w_{c}\' for c in LABEL_COLS]\n        self.w = (self.df[weight_cols].to_numpy(np.float32) if set(weight_cols) <= set(self.df)\n                  else np.ones_like(self.y))\n        self.present = self.df[[f"present_{s}" for s in range(len(PLANES))]].to_numpy(bool)\n        self.row_of = self.df["row"].to_numpy(int)\n        self.shard_of = (self.df["cache_id"].to_numpy(int) if "cache_id" in self.df\n                         else np.zeros(len(self.df), int))\n        self.caches = {}                       # memmaps opened lazily, per worker and shard\n        # Training batches carry the report-embedding targets after the labels (weight 1 where known).\n        rep = globals().get(\'REP_COLS\') or []\n        self.aux = self.df[rep].to_numpy(np.float32) if (train and rep) else None\n\n    def _cache(self, shard):\n        if shard not in self.caches:\n            self.caches[shard] = np.load(os.path.join(CACHE_DIRS[shard], "cache.npy"), mmap_mode="r")\n        return self.caches[shard]\n\n    def __len__(self):\n        return len(self.rows)\n\n    def _aug(self, vol):\n        # NO horizontal or vertical flip: a horizontal flip in the coronal plane swaps\n        # medial and lateral, which would invert four of the twelve labels.\n        h, w = vol.shape[-2:]\n        ang = np.random.uniform(-12, 12)\n        sc  = np.random.uniform(0.9, 1.1)\n        M = cv2.getRotationMatrix2D((w/2, h/2), ang, sc)\n        M[:, 2] += np.random.uniform(-0.06, 0.06, 2) * w\n        out = np.empty_like(vol)\n        for i in range(vol.shape[0]):\n            out[i] = cv2.warpAffine(vol[i], M, (w, h), flags=cv2.INTER_LINEAR,\n                                    borderValue=0)\n        if np.random.rand() < 0.5:\n            out = np.clip(out.astype(np.float32) * np.random.uniform(0.85, 1.15)\n                          + np.random.uniform(-15, 15), 0, 255).astype(np.uint8)\n        return out\n\n    def _aug_slot(self, vol):\n        # Public-member jitter for slot_query: small rotation, zoom-in only (zooming out\n        # would invent tissue at the crop edge, where the popliteal fossa sits), edge\n        # replication, multiplicative intensity. Still no flips.\n        h, w = vol.shape[-2:]\n        M = cv2.getRotationMatrix2D((w/2, h/2), np.random.uniform(-8, 8),\n                                    np.random.uniform(1.0, 1.08))\n        M[:, 2] += np.random.uniform(-0.025, 0.025, 2) * w\n        out = np.stack([cv2.warpAffine(s, M, (w, h), flags=cv2.INTER_LINEAR,\n                                       borderMode=cv2.BORDER_REPLICATE) for s in vol])\n        return np.clip(out.astype(np.float32) * np.random.uniform(0.9, 1.1),\n                       0, 255).astype(np.uint8)\n\n    def __getitem__(self, i):\n        j = self.rows[i]\n        r = self.row_of[j]\n        cache = self._cache(self.shard_of[j])\n        present = self.present[j]\n\n        if self.train and ARCH == \'reader25d\':\n            # All windows; the model subsamples windows and augments each series on GPU.\n            vol = np.asarray(cache[r])\n        elif self.train and ARCH == \'slot_query\':\n            # One slice from each third of the band, drawn independently per slot, so\n            # every cached position trains while each input still spans the joint.\n            parts = slice_thirds(cache.shape[2])\n            vol = np.stack([self._aug_slot(np.asarray(cache[r, s, [np.random.choice(p) for p in parts]]))\n                            if present[s] else np.zeros((3, *cache.shape[3:]), np.uint8)\n                            for s in range(cache.shape[1])])\n        elif self.train:\n            # Slice the memmap BEFORE materialising, so the discarded slices are\n            # never read off disk at all (~37% less I/O per sample).\n            avail = cache.shape[2]\n            t = min(T_TRAIN, avail)\n            t0 = np.random.randint(0, avail - t + 1)\n            vol = np.asarray(cache[r][:, t0:t0 + t])\n            vol = np.stack([self._aug(v) if present[s] else v\n                            for s, v in enumerate(vol)])\n        else:\n            vol = np.asarray(cache[r])               # eval uses every slice\n\n        y, w = self.y[j], self.w[j]\n        if self.aux is not None:\n            y = np.concatenate([y, self.aux[j]])\n            w = np.concatenate([w, np.isfinite(self.aux[j]).astype(np.float32)])\n        return (torch.from_numpy(np.array(vol, copy=True, order=\'C\')),\n                torch.from_numpy(present.copy()),\n                torch.from_numpy(y.copy()),\n                torch.from_numpy(w.copy()))\n\n\ndef collate(batch):\n    xs, ps, ys, ws = zip(*batch)\n    return torch.stack(xs), torch.stack(ps), torch.stack(ys), torch.stack(ws)\n\n"""Shared model architecture for training and offline inference."""\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport timm\n\ndef make_neighbours(x):\n    # (B,S,T,H,W) -> (B,S,T,3,H,W) from slices t-1,t,t+1 (edges clamped)\n    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)\n    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)\n    return torch.stack([prev, x, nxt], dim=3)\n\n\nclass AttnPool(nn.Module):\n    # Attention over slices within one plane.\n    def __init__(self, dim, hidden=128):\n        super().__init__()\n        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\n\n    def forward(self, x, mask=None):               # x (B,T,D)\n        a = self.score(x).squeeze(-1)\n        if mask is not None:\n            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)\n        a = a.softmax(dim=-1)\n        return (x * a.unsqueeze(-1)).sum(1), a\n\n\nclass LabelAttnPool(nn.Module):\n    # Per-label attention over planes: each finding picks the view it needs.\n    def __init__(self, dim, n_labels):\n        super().__init__()\n        self.score = nn.Linear(dim, n_labels)\n\n    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)\n        a = self.score(x)                          # (B,S,L)\n        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)\n        a = a.softmax(dim=1)                       # over planes, per label\n        z = torch.einsum("bsd,bsl->bld", x, a)     # (B,L,D)\n        return z, a\n\n\nclass KneeNet(nn.Module):\n    def __init__(self, backbone="tf_efficientnet_b0.ns_jft_in1k", n_out=12, n_planes=3,\n                 pretrained=True, drop=0.3, img=224, normalization="legacy",\n                 encoder_chunk=64):\n        super().__init__()\n        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}\n        self.enc = timm.create_model(backbone, pretrained=pretrained,\n                                     num_classes=0, in_chans=3, **kwargs)\n        self.encoder_chunk = encoder_chunk\n        config = timm.data.resolve_model_data_config(self.enc)\n        mean = config[\'mean\'] if normalization == \'pretrained\' else (0.449,)*3\n        std = config[\'std\'] if normalization == \'pretrained\' else (0.226,)*3\n        self.register_buffer(\'input_mean\', torch.tensor(mean).view(1,3,1,1), persistent=False)\n        self.register_buffer(\'input_std\', torch.tensor(std).view(1,3,1,1), persistent=False)\n        d = self.enc.num_features\n        self.slice_pool = AttnPool(d)\n        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))\n        self.plane_pool = LabelAttnPool(d, n_out)\n        self.drop = nn.Dropout(drop)\n        self.head_w = nn.Parameter(torch.zeros(n_out, d))\n        self.head_b = nn.Parameter(torch.zeros(n_out))\n        nn.init.trunc_normal_(self.head_w, std=0.02)\n\n    def forward(self, x, present):\n        # autocast entered HERE, not in the training loop: it is thread-local, so a\n        # loop-level autocast never reaches a threaded forward. Verified reaching the\n        # encoder as float16.\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            B, S, T, H, W = x.shape\n            x = x.float().div(255.)   # ImageNet grey stats\n            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)\n\n            x3 = (x3 - self.input_mean) / self.input_std\n            # Chunk evaluation to bound memory; training keeps BatchNorm batch statistics intact.\n            if not self.training and self.encoder_chunk and len(x3) > self.encoder_chunk:\n                features = torch.cat([self.enc(part) for part in x3.split(self.encoder_chunk)])\n            else:\n                features = self.enc(x3)\n            f = features.reshape(B * S, T, -1)\n            v, _ = self.slice_pool(f)\n            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)\n\n            m = present.clone()\n            m[~m.any(dim=1)] = True                # guard: never all-masked\n            z, plane_att = self.plane_pool(v, m)\n            z = self.drop(z)\n            logits = (z * self.head_w.unsqueeze(0)).sum(-1) + self.head_b\n        return logits, plane_att\n\n\n\ndef slice_thirds(n):\n    # Lower, middle and upper thirds of the cached slice band.\n    parts = np.array_split(np.arange(n), 3)\n    if min(len(p) for p in parts) == 0:\n        return [np.array([0]), np.array([(n - 1) // 2]), np.array([n - 1])]\n    return parts\n\n\ndef spread_groups(n):\n    # 3-slice inputs that each span the band: one slice from every third. For six cached\n    # slices that is [0, 2, 4] and [1, 3, 5]; a 3-slice input is its own single group.\n    parts = slice_thirds(n)\n    return [[int(p[g]) for p in parts] for g in range(min(len(p) for p in parts))]\n\n\nclass SlotQueryHead(nn.Module):\n    # Per-finding query attention over series slots (public DINOv2 member head).\n    def __init__(self, dim, n_slots, n_out, hidden=256, drop=0.2):\n        super().__init__()\n        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())\n        self.slot_emb = nn.Parameter(torch.randn(n_slots, hidden) * 0.02)\n        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)\n        self.drop = nn.Dropout(drop)\n        self.out = nn.Linear(hidden, n_out)\n        self.hidden = hidden\n\n    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)\n        h = self.proj(x) + self.slot_emb\n        a = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5\n        a = a.masked_fill(~mask.unsqueeze(1), torch.finfo(a.dtype).min).softmax(-1)\n        z = self.drop(torch.einsum("bos,bsh->boh", a, h))\n        return (z * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias, a\n\n\nclass SlotQueryNet(nn.Module):\n    """One 3-slice image per series slot, encoder partly fine-tuned, query head on top.\n\n    Training feeds one spread triplet per slot (T=3). Evaluation feeds every cached slice\n    and averages logits over the spread groups, so no slice position is left unseen.\n    """\n    def __init__(self, backbone="vit_small_patch14_dinov2.lvd142m", n_out=12, n_slots=6,\n                 pretrained=True, img=336, drop=0.2, hidden=256, unfreeze_last=6,\n                 normalization="pretrained", encoder_chunk=48):\n        super().__init__()\n        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}\n        self.enc = timm.create_model(backbone, pretrained=pretrained,\n                                     num_classes=0, in_chans=3, **kwargs)\n        self.img = img\n        self.encoder_chunk = encoder_chunk\n        self.tokens = hasattr(self.enc, "blocks") and hasattr(self.enc, "num_prefix_tokens")\n        config = timm.data.resolve_model_data_config(self.enc)\n        mean = config[\'mean\'] if normalization == \'pretrained\' else (0.449,)*3\n        std = config[\'std\'] if normalization == \'pretrained\' else (0.226,)*3\n        self.register_buffer(\'input_mean\', torch.tensor(mean).view(1,3,1,1), persistent=False)\n        self.register_buffer(\'input_std\', torch.tensor(std).view(1,3,1,1), persistent=False)\n        if unfreeze_last is not None and hasattr(self.enc, "blocks"):\n            # Early blocks stay at their self-supervised weights; only late blocks adapt.\n            for p in self.enc.parameters(): p.requires_grad = False\n            blocks = list(self.enc.blocks)\n            for block in blocks[max(0, len(blocks) - unfreeze_last):]:\n                for p in block.parameters(): p.requires_grad = True\n            for name in ("norm", "fc_norm"):\n                module = getattr(self.enc, name, None)\n                if isinstance(module, nn.Module):\n                    for p in module.parameters(): p.requires_grad = True\n        dim = self.enc.num_features * 2\n        self.head = SlotQueryHead(dim, n_slots, n_out, hidden=hidden, drop=drop)\n\n    def encode(self, x):                           # (N,3,H,W) normalised -> (N,2D)\n        f = self.enc.forward_features(x)\n        if self.tokens:\n            # CLS token plus mean of patch tokens; register tokens are excluded.\n            return torch.cat([f[:, 0], f[:, self.enc.num_prefix_tokens:].mean(1)], dim=1)\n        if f.ndim == 4:\n            return torch.cat([f.mean((-2, -1)), f.amax((-2, -1))], dim=1)\n        return torch.cat([f, f], dim=1)\n\n    def forward(self, x, present):\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            B, S, T, H, W = x.shape\n            m = present.clone()\n            m[~m.any(dim=1)] = True                # guard: never all-masked\n            keep = m.reshape(-1)\n            groups = spread_groups(T)\n            logits, attention = 0., 0.\n            for group in groups:\n                images = x[:, :, group].reshape(B * S, 3, H, W)[keep].float().div(255.)\n                if H != self.img:\n                    images = nn.functional.interpolate(images, size=(self.img, self.img),\n                                                       mode="bilinear", align_corners=False)\n                images = (images - self.input_mean) / self.input_std\n                # Absent slots are never encoded; their features stay zero and masked.\n                if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:\n                    f = torch.cat([self.encode(part) for part in images.split(self.encoder_chunk)])\n                else:\n                    f = self.encode(images)\n                features = f.new_zeros(B * S, f.shape[-1])\n                features[keep] = f\n                z, a = self.head(features.reshape(B, S, -1), m)\n                logits, attention = logits + z, attention + a\n        return logits / len(groups), attention / len(groups)\n\n\nclass Reader25D(nn.Module):\n    """2.5D study reader (public ConvNeXt reader layout).\n\n    Every cached window of three adjacent slices is one CNN image. Its feature becomes a\n    token tagged with its series slot and its position along the stack; a small\n    transformer mixes all tokens of the study and each finding pools them with its own\n    attention. Training augments each series with one shared rigid/intensity jitter (no\n    flips: laterality is already normalised) and encodes a random subset of windows.\n    """\n    def __init__(self, backbone="convnext_tiny.in12k_ft_in1k", n_out=12, n_slots=6,\n                 pretrained=True, img=240, dim=384, depth=2, drop=0.2, drop_path=0.1,\n                 normalization="pretrained", band=(0.06, 0.94), train_windows=6,\n                 encoder_chunk=96, slot_dropout=0.0, aux_dim=0):\n        super().__init__()\n        self.slot_dropout = slot_dropout     # training only: hide whole series (protocols vary by site)\n        # Runtime switches for XLA/TPU training; neither changes the parameters or the checkpoint.\n        # static_shapes: encode a fixed number of windows for every slot (absent ones masked), so\n        # tensor shapes never depend on the data. augment_in_model=False: the data loader already\n        # applied the per-series jitter (augment_series_cpu), so the model only normalises.\n        self.static_shapes = False\n        self.augment_in_model = True\n        # Test-time window subsets (eval only; inference WINDOW_TTA): encode a random subset of this many\n        # windows per series, as in training, instead of all of them. None: every window.\n        self.subset_windows = None\n        kwargs = {"drop_path_rate": drop_path} if drop_path else {}\n        if backbone.startswith("vit_"):\n            kwargs["img_size"] = img\n        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kwargs)\n        self.img, self.band, self.train_windows = img, tuple(band), train_windows\n        self.encoder_chunk = encoder_chunk\n        config = timm.data.resolve_model_data_config(self.enc)\n        mean = config[\'mean\'] if normalization == \'pretrained\' else (0.449,)*3\n        std = config[\'std\'] if normalization == \'pretrained\' else (0.226,)*3\n        self.register_buffer(\'input_mean\', torch.tensor(mean).view(1,3,1,1), persistent=False)\n        self.register_buffer(\'input_std\', torch.tensor(std).view(1,3,1,1), persistent=False)\n        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim),\n                                  nn.LayerNorm(dim))\n        self.slot_emb = nn.Embedding(n_slots, dim)\n        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))\n        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True,\n                                           norm_first=True)\n        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None\n        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                 nn.Linear(256, n_out))\n        self.cls_w = nn.Parameter(torch.randn(n_out, dim) * 0.02)\n        self.cls_b = nn.Parameter(torch.zeros(n_out))\n        # Report-grounded auxiliary head (training only): regresses the study\'s report embedding\n        # from the finding-pooled and the mean study tokens. Its output is kept in last_aux; the\n        # logits and the inference path do not depend on it.\n        self.aux = (nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU(), nn.Linear(dim, aux_dim))\n                    if aux_dim else None)\n        self.last_aux = None\n\n    def _aux(self, h, pad, pooled):\n        if self.aux is None:\n            return\n        keep = (~pad).unsqueeze(-1).to(h.dtype)\n        mean = (h * keep).sum(1) / keep.sum(1).clamp_min(1)\n        self.last_aux = self.aux(torch.cat([pooled.mean(1), mean], -1))\n\n    def views(self, x, series, n_series):\n        # x float [N,3,H,W] in 0..1; one random transform per series in training.\n        if x.shape[-1] != self.img:\n            x = nn.functional.interpolate(x, size=(self.img, self.img), mode="bilinear",\n                                          align_corners=False, antialias=x.shape[-1] > self.img)\n        if self.training and self.augment_in_model:\n            dev = x.device\n            ang = (torch.rand(n_series, device=dev) - .5) * 2 * np.radians(12)\n            sc = 1 + (torch.rand(n_series, device=dev) - .5) * .24\n            tx, ty = ((torch.rand(2, n_series, device=dev) - .5) * .12)\n            theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),\n                                 torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)\n            grid = nn.functional.affine_grid(theta[series], x.shape, align_corners=False)\n            x = nn.functional.grid_sample(x, grid, mode="bilinear", padding_mode="zeros",\n                                          align_corners=False)\n            gamma = torch.exp((torch.rand(n_series, device=dev) - .5) * .5)[series]\n            gain = (1 + (torch.rand(n_series, device=dev) - .5) * .2)[series]\n            x = x.clamp_min(1e-4).pow(gamma.view(-1, 1, 1, 1)) * gain.view(-1, 1, 1, 1)\n        return (x - self.input_mean) / self.input_std\n\n    def forward(self, x, present):\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            B, S, T, H, W = x.shape\n            K = T // 3\n            m = present.clone()\n            if self.training and self.slot_dropout > 0:\n                kept = m & (torch.rand(m.shape, device=m.device) >= self.slot_dropout)\n                # A study that lost every series keeps one of its present series, chosen at random.\n                pick = torch.rand(m.shape, device=m.device).masked_fill(~m, -1).argmax(dim=1)\n                lost = ~kept.any(dim=1) & m.any(dim=1)\n                one = nn.functional.one_hot(pick, m.shape[1]).bool()\n                m = kept | (lost.unsqueeze(1) & one)   # fixed-shape boolean algebra (XLA-safe)\n            m = m | ~m.any(dim=1, keepdim=True)    # guard: never all-masked\n            if self.static_shapes:\n                return self._forward_static(x, m, B, S, K, H, W)\n            tokens = m.unsqueeze(-1).expand(B, S, K).clone()\n            n_win = self.train_windows if self.training else getattr(self, \'subset_windows\', None)\n            if n_win and n_win < K:\n                # Encode a random subset of windows per series: stack augmentation that\n                # also bounds activation memory. Positions stay exact.\n                rank = torch.rand(B, S, K, device=x.device).argsort(-1).argsort(-1)\n                tokens &= rank < n_win\n            keep = tokens.reshape(-1)\n            series = torch.arange(B * S, device=x.device).repeat_interleave(K)[keep]\n            images = x[:, :, :3 * K].reshape(B * S * K, 3, H, W)[keep].float().div(255.)\n            images = self.views(images, series, B * S).contiguous(memory_format=torch.channels_last)\n            if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:\n                f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])\n            else:\n                f = self.enc(images)\n            feat = f.new_zeros(B * S * K, f.shape[-1])\n            feat[keep] = f\n            slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, K)\n            pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)\n            h = (self.proj(feat).view(B, S, K, -1) + self.slot_emb(slot)\n                 + self.pos_emb(pos.view(1, 1, K, 1).to(feat.dtype)))\n            h = h.reshape(B, S * K, -1)\n            pad = ~tokens.reshape(B, S * K)\n            if self.tf is not None:\n                h = self.tf(h, src_key_padding_mask=pad)\n            a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)\n            pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)\n            logits = (pooled * self.cls_w).sum(-1) + self.cls_b\n            self._aux(h, pad, pooled)\n        return logits, a\n\n\ndef _reader_forward_static(self, x, m, B, S, K, H, W):\n    """Reader25D forward with data-independent shapes (XLA/TPU). Every slot encodes the same\n    number of windows; absent or dropped slots are encoded too but masked, exactly as the\n    dynamic path masks them, so outputs agree up to floating-point error."""\n    n_win = self.train_windows if self.training else getattr(self, \'subset_windows\', None)\n    n_win = n_win if n_win and n_win < K else K\n    if n_win < K:\n        idx = torch.rand(B, S, K, device=x.device).argsort(-1)[..., :n_win].sort(-1).values\n    else:\n        idx = torch.arange(K, device=x.device).view(1, 1, K).expand(B, S, K)\n    windows = x[:, :, :3 * K].reshape(B, S, K, 3, H, W)\n    windows = torch.gather(windows, 2, idx.reshape(B, S, n_win, 1, 1, 1).expand(B, S, n_win, 3, H, W))\n    images = windows.reshape(B * S * n_win, 3, H, W).float().div(255.)\n    series = torch.arange(B * S, device=x.device).repeat_interleave(n_win)\n    images = self.views(images, series, B * S)\n    if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:\n        f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])\n    else:\n        f = self.enc(images)\n    tokens = m.unsqueeze(-1).expand(B, S, n_win)\n    slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, n_win)\n    pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)[idx]\n    h = (self.proj(f).view(B, S, n_win, -1) + self.slot_emb(slot)\n         + self.pos_emb(pos.unsqueeze(-1).to(f.dtype)))\n    h = h.reshape(B, S * n_win, -1)\n    pad = ~tokens.reshape(B, S * n_win)\n    if self.tf is not None:\n        h = self.tf(h, src_key_padding_mask=pad)\n    a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)\n    pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)\n    logits = (pooled * self.cls_w).sum(-1) + self.cls_b\n    self._aux(h, pad, pooled)\n    return logits, a\n\n\nReader25D._forward_static = _reader_forward_static\n\n\ndef augment_series_cpu(x, generator=None):\n    """Reader25D\'s training jitter on the CPU, for loaders that feed XLA devices: one rotation\n    (+-12 deg), scale (+-12%), shift (+-6%), gamma and gain per series, shared by all its slices.\n    x: uint8 [S, T, H, W] of one study; returns uint8 of the same shape (values clipped to 0..255)."""\n    S, T, H, W = x.shape\n    rnd = lambda *shape: torch.rand(*shape, generator=generator)\n    ang = (rnd(S) - .5) * 2 * np.radians(12)\n    sc = 1 + (rnd(S) - .5) * .24\n    tx, ty = (rnd(2, S) - .5) * .12\n    theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),\n                         torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)\n    xf = x.float().div(255.)\n    grid = nn.functional.affine_grid(theta, (S, T, H, W), align_corners=False)\n    xf = nn.functional.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n    gamma = torch.exp((rnd(S) - .5) * .5).view(S, 1, 1, 1)\n    gain = (1 + (rnd(S) - .5) * .2).view(S, 1, 1, 1)\n    xf = xf.clamp_min(1e-4).pow(gamma) * gain\n    return (xf.clamp(0, 1) * 255).round().to(torch.uint8)\n\n\ndef window_centres(n_slices):\n    # Centre slice of each cached 3-slice window (v3 layout); plain slices otherwise.\n    return list(range(1, n_slices, 3)) if n_slices % 3 == 0 and n_slices >= 6 else list(range(n_slices))\n\n\nclass RadImageNetHead(nn.Module):\n    """Frozen official RadImageNet ResNet-50 slice encoder + per-finding query head.\n\n    Follows the public RadImageNet arm (E9/E11): every window-centre slice is one\n    grey image scaled to [-1, 1]; its 2048-d global-average feature is a token tagged\n    with slot and stack position; one query per finding cross-attends over all tokens\n    and is fused with the mean token. Only the head trains.\n    """\n    def __init__(self, n_out=12, n_slots=6, n_windows=8, img=224, head_dim=512,\n                 weights_path=None, encoder_chunk=128):\n        super().__init__()\n        from torchvision.models import resnet50\n        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])\n        if weights_path:\n            state = torch.load(weights_path, map_location="cpu", weights_only=True)\n            self.backbone.load_state_dict({k.split("backbone.", 1)[1]: v for k, v in state.items()},\n                                          strict=True)\n        for p in self.backbone.parameters(): p.requires_grad = False\n        self.img, self.n_windows, self.encoder_chunk = img, n_windows, encoder_chunk\n        dim = 2048\n        self.project = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, head_dim), nn.GELU())\n        self.plane = nn.Parameter(torch.randn(n_slots, head_dim) * .01)\n        self.position = nn.Parameter(torch.randn(n_windows, head_dim) * .01)\n        self.query = nn.Parameter(torch.randn(n_out, head_dim) * .02)\n        self.attn = nn.MultiheadAttention(head_dim, 8, dropout=.10, batch_first=True)\n        self.fuse = nn.Sequential(nn.LayerNorm(head_dim * 4), nn.Linear(head_dim * 4, head_dim),\n                                  nn.GELU(), nn.Dropout(.15))\n        self.weight = nn.Parameter(torch.randn(n_out, head_dim) * .02)\n        self.bias = nn.Parameter(torch.zeros(n_out))\n\n    def train(self, mode=True):\n        super().train(mode)\n        self.backbone.eval()                       # frozen BatchNorm statistics\n        return self\n\n    @torch.no_grad()\n    def encode(self, x, present):\n        # x uint8 (B,S,T,H,W) -> tokens (B, S*K, 2048) float16, token mask (B, S*K)\n        B, S, T, H, W = x.shape\n        centres = window_centres(T)[:self.n_windows]\n        K = len(centres)\n        keep = present.unsqueeze(-1).expand(B, S, K).reshape(-1)\n        images = x[:, :, centres].reshape(B * S * K, 1, H, W)[keep].float().div(127.5).sub(1.)\n        if H != self.img:\n            images = nn.functional.interpolate(images, size=(self.img, self.img), mode="bilinear",\n                                               align_corners=False)\n        images = images.expand(-1, 3, -1, -1)\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            f = torch.cat([self.backbone(part).mean(dim=(2, 3)) for part in images.split(self.encoder_chunk)]) \\\n                if len(images) else images.new_zeros(0, 2048)\n        tokens = torch.zeros(B * S * K, 2048, device=x.device, dtype=torch.float16)\n        tokens[keep] = f.to(torch.float16)\n        return tokens.view(B, S * K, 2048), keep.view(B, S * K).float()\n\n    def head(self, feature, mask):\n        n_slots = self.plane.shape[0]\n        token = self.project(feature.float())\n        token = token.view(len(token), n_slots, -1, token.shape[-1])\n        token = token + self.plane[None, :, None] + self.position[None, None, :token.shape[2]]\n        token = token.flatten(1, 2)\n        pad = mask <= 0\n        empty = pad.all(1)\n        if empty.any():\n            pad = pad.clone(); pad[empty, 0] = False\n        query = self.query.unsqueeze(0).expand(len(token), -1, -1)\n        attended = query + self.attn(query, token, token, key_padding_mask=pad, need_weights=False)[0]\n        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)\n        mean = mean.expand(-1, len(self.query), -1)\n        fused = self.fuse(torch.cat([attended, mean, (attended - mean).abs(), attended * mean], -1))\n        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias\n\n    def forward(self, x, present):\n        m = present.clone()\n        m[~m.any(dim=1)] = True\n        return self.head(*self.encode(x, m)), None\n\n\ndef build_model(cfg, n_out, n_planes, pretrained=True):\n    # A checkpoint\'s recipe rebuilds its own architecture; old recipes have no arch key.\n    arch = cfg.get(\'arch\', \'kneenet\')\n    if arch == \'kneenet\':\n        return KneeNet(cfg[\'backbone\'], n_out, n_planes, pretrained=pretrained,\n                       drop=cfg.get(\'drop\', .3), img=cfg[\'img\'],\n                       normalization=cfg.get(\'normalization\', \'legacy\'))\n    if arch == \'slot_query\':\n        return SlotQueryNet(cfg[\'backbone\'], n_out, n_planes, pretrained=pretrained,\n                            img=cfg[\'img\'], drop=cfg.get(\'drop\', .2),\n                            hidden=cfg.get(\'head_hidden\', 256),\n                            unfreeze_last=cfg.get(\'unfreeze_last\', 6),\n                            normalization=cfg.get(\'normalization\', \'pretrained\'))\n    if arch == \'reader25d\':\n        return Reader25D(cfg[\'backbone\'], n_out, n_planes, pretrained=pretrained,\n                         img=cfg[\'img\'], drop=cfg.get(\'drop\', .2),\n                         dim=cfg.get(\'head_hidden\', 384), drop_path=cfg.get(\'drop_path\', .1),\n                         band=tuple(cfg[\'preprocessing\'][\'slice_band\']),\n                         train_windows=cfg.get(\'train_windows\', 6),\n                         normalization=cfg.get(\'normalization\', \'pretrained\'),\n                         slot_dropout=cfg.get(\'slot_dropout\', 0.0), aux_dim=cfg.get(\'aux_dim\', 0))\n    if arch == \'radhead\':\n        # The frozen encoder ships inside every checkpoint; weights_path is training-only.\n        return RadImageNetHead(n_out, n_planes, n_windows=cfg.get(\'n_windows\', 8), img=cfg[\'img\'],\n                               head_dim=cfg.get(\'head_hidden\', 512),\n                               weights_path=cfg.get(\'weights_path\') if pretrained else None)\n    raise ValueError(f\'Unknown architecture {arch!r}\')\n\ndef masked_bce(logits, targets, weights=None):\n    # BCE over (B,12) soft targets that may contain NaN. NaN cells get exactly zero\n    # gradient, so a study a reader abstained on still trains the other heads. Optional\n    # per-cell weights up-weight gold rows; the mean is taken over observed weight.\n    mask = ~torch.isnan(targets)\n    if not mask.any():\n        return logits.sum() * 0.0\n    w = mask.float() if weights is None else weights * mask\n    loss = F.binary_cross_entropy_with_logits(\n        logits, torch.nan_to_num(targets, nan=0.0), reduction="none")\n    return (loss * w).sum() / w.sum().clamp_min(1e-6)\n\n"""Five-fold training runtime, embedded in training.ipynb."""\ndef seed_worker(worker_id):\n    seed = torch.initial_seed() % 2**32\n    np.random.seed(seed); random.seed(seed)\n    cv2.setNumThreads(0)\n\n\ndef make_loader(rows, training, seed):\n    return DataLoader(KneeDS(df, rows, training), batch_size=BATCH,\n                      shuffle=training, num_workers=NUM_WORKERS,\n                      collate_fn=collate, pin_memory=DEV==\'cuda\', drop_last=False,\n                      persistent_workers=NUM_WORKERS > 0, worker_init_fn=seed_worker,\n                      generator=torch.Generator().manual_seed(seed),\n                      **({\'prefetch_factor\': 2} if NUM_WORKERS else {}))\n\n\n@torch.inference_mode()\ndef evaluate(model, loader):\n    model.eval()\n    ys, ps = [], []\n    for x, present, y, *_ in loader:\n        logits, _ = model(x.to(DEV, non_blocking=True), present.to(DEV, non_blocking=True))\n        ps.append(logits.float().sigmoid().cpu().numpy()); ys.append(y.numpy())\n    if not ys: return np.empty((0,len(LABEL_COLS))), np.empty((0,len(LABEL_COLS)))\n    return np.concatenate(ys), np.concatenate(ps)\n\n\ndef atomic_torch_save(obj, path):\n    temp = str(path) + \'.tmp\'\n    torch.save(obj, temp); os.replace(temp, path)\n\n\nos.makedirs(OUT, exist_ok=True)\nfolds = assign_folds(df, LABEL_COLS, N_FOLDS, SEED, GROUP_COL if GROUP_COL in df else None)\nfold_table = df[[\'StudyInstanceUID\', \'source\']].assign(fold=folds)\nwrite_atomic(os.path.join(OUT, \'folds.csv\'), lambda p: fold_table.to_csv(p, index=False))\n# Record the dataset identity so resumed folds cannot silently use changed labels or cache.\nimport hashlib, importlib.metadata\nidentity = hashlib.sha256((df.to_json(orient=\'split\') + json.dumps(CACHE_META, sort_keys=True)\n                           + fold_table.to_json()).encode()).hexdigest()\nrecipe = dict(arch=ARCH, backbone=BACKBONE, img=IMG, n_slices=N_SLICES, planes=PLANES,\n              label_cols=LABEL_COLS, t_train=T_TRAIN, preprocessing=CACHE_META,\n              unfreeze_last=UNFREEZE_LAST, head_hidden=HEAD_HIDDEN, pct_start=PCT_START,\n              normalization=NORMALIZATION, drop=DROPOUT, seed=SEED,\n              epochs=EPOCHS, lr=LR, weight_decay=WD, n_folds=N_FOLDS, accum_steps=ACCUM_STEPS,\n              encoder_lr_scale=ENCODER_LR_SCALE, batch=BATCH, identity=identity,\n              gold_weight=GOLD_WEIGHT, ema=EMA_DECAY, clip=CLIP, train_windows=TRAIN_WINDOWS, slot_dropout=SLOT_DROPOUT,\n              preset=PRESET, grad_checkpoint=GRAD_CHECKPOINT, teacher=TEACHER_SUMMARY,\n              group_col=GROUP_COL if GROUP_COL in df else None,\n              aux_dim=globals().get(\'AUX_DIM\', 0), report_aux=globals().get(\'REPORT_AUX_SUMMARY\'),\n              versions={p: importlib.metadata.version(p) for p in (\'torch\',\'timm\',\'numpy\')})\nwrite_atomic(os.path.join(OUT, \'run_config.json\'), json.dumps(recipe, indent=2))\nREPORT_AUX_WEIGHT = globals().get(\'REPORT_AUX\', 0.0) if globals().get(\'AUX_DIM\', 0) else 0.0\n\n\ndef report_aux_loss(pred, target, known):\n    # Squared error to the report embedding (components scaled to mean variance 1), rows with one only.\n    rows = known.all(1)\n    if not rows.any():\n        return pred.sum() * 0\n    return ((pred[rows].float() - target[rows]) ** 2).mean()\n\n\nall_hist = []\nfor fold in FOLDS_TO_RUN:\n    assert fold in range(N_FOLDS)\n    fold_seed = SEED + fold\n    torch.manual_seed(fold_seed); np.random.seed(fold_seed); random.seed(fold_seed)\n    # Gold studies never validate; with GOLD_WEIGHT > 0 they train in every fold.\n    tr_rows = np.flatnonzero(((folds >= 0) & (folds != fold)) | ((folds == -1) & (GOLD_WEIGHT > 0)))\n    va_rows = np.flatnonzero(folds == fold)\n    assert not set(tr_rows) & set(va_rows)\n    assert df.iloc[va_rows][\'source\'].eq(\'pseudo\').all()\n    assert GOLD_WEIGHT > 0 or df.iloc[tr_rows][\'source\'].eq(\'pseudo\').all()\n    dl_tr = make_loader(tr_rows, True, fold_seed)\n    dl_val = make_loader(va_rows, False, fold_seed)\n    model = build_model(recipe, len(LABEL_COLS), len(PLANES),\n                        pretrained=not (RESUME and os.path.exists(os.path.join(OUT, f\'last_fold_{fold}.pt\')))).to(DEV)\n    if GRAD_CHECKPOINT and hasattr(model.enc, \'set_grad_checkpointing\'):\n        model.enc.set_grad_checkpointing(True)     # recompute activations: fits large backbones on a T4\n    # Frozen encoder blocks are left out of the optimizer entirely.\n    encoder_params = [p for p in model.enc.parameters() if p.requires_grad]\n    head_params = [p for name,p in model.named_parameters() if not name.startswith(\'enc.\')]\n    print(f\'trainable: encoder {sum(p.numel() for p in encoder_params)/1e6:.1f}M, \'\n          f\'head {sum(p.numel() for p in head_params)/1e6:.2f}M\')\n    opt = torch.optim.AdamW([{\'params\':encoder_params,\'lr\':LR*ENCODER_LR_SCALE},\n                            {\'params\':head_params,\'lr\':LR}], weight_decay=WD)\n    total_steps = EPOCHS*math.ceil(len(dl_tr)/ACCUM_STEPS)\n    pct_start = PCT_START\n    if abs(pct_start * total_steps - 1) < 1e-6:\n        # OneCycleLR divides by zero when the warmup is exactly one optimizer step.\n        pct_start = 1.5 / total_steps if total_steps >= 2 else .5\n    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR*ENCODER_LR_SCALE, LR], total_steps=total_steps,\n                    pct_start=pct_start)\n    scaler = torch.amp.GradScaler(DEV, enabled=DEV==\'cuda\')\n    # Exponential moving average of the weights; it is what validates and what is saved.\n    ema, ema_steps = None, 0\n    if EMA_DECAY:\n        ema = copy.deepcopy(model).eval()\n        for p in ema.parameters(): p.requires_grad_(False)\n    path = os.path.join(OUT, f\'fold_{fold}.pt\')\n    last_path = os.path.join(OUT, f\'last_fold_{fold}.pt\')\n    start, best, hist = 0, float(\'inf\'), []\n    if RESUME and os.path.exists(last_path):\n        saved = torch.load(last_path, map_location=\'cpu\', weights_only=False)\n        assert saved[\'cfg\'] == recipe, \'Resume recipe differs; use a new OUT directory\'\n        model.load_state_dict(saved[\'model\']); opt.load_state_dict(saved[\'optimizer\'])\n        sched.load_state_dict(saved[\'scheduler\']); scaler.load_state_dict(saved[\'scaler\'])\n        start, best, hist = saved[\'epoch\']+1, saved[\'best\'], saved[\'history\']\n        if ema is not None:\n            ema.load_state_dict(saved[\'ema\']); ema_steps = saved[\'ema_steps\']\n        assert os.path.exists(path), \'Resume requires both last and best fold checkpoints\'\n    elif os.path.exists(path):\n        raise RuntimeError(f\'{path} already exists; set RESUME=True or choose an empty OUT\')\n    for ep in range(start, EPOCHS):\n        # Epoch-local worker seeds make interrupted runs repeatable at epoch boundaries.\n        epoch_seed = fold_seed*1000 + ep\n        torch.manual_seed(epoch_seed); np.random.seed(epoch_seed); random.seed(epoch_seed)\n        dl_tr = make_loader(tr_rows, True, epoch_seed)\n        model.train(); total, count, aux_total = 0., 0, 0.; t0 = time.time()\n        opt.zero_grad(set_to_none=True)\n        for step, (x, present, y, w) in enumerate(tqdm(dl_tr, desc=f\'fold {fold} epoch {ep+1}\')):\n            x, present, y, w = x.to(DEV, non_blocking=True), present.to(DEV), y.to(DEV), w.to(DEV)\n            n_lab = len(LABEL_COLS)\n            y_aux, w_aux, y, w = y[:, n_lab:], w[:, n_lab:], y[:, :n_lab], w[:, :n_lab]\n            logits, _ = model(x, present)\n            loss = masked_bce(logits.float(), y, w)\n            if REPORT_AUX_WEIGHT:\n                aux = report_aux_loss(model.last_aux, y_aux, w_aux > 0)\n                loss = loss + REPORT_AUX_WEIGHT * aux\n                aux_total += aux.item() * len(x)\n            if not torch.isfinite(loss): raise FloatingPointError(\'Non-finite loss\')\n            group_start = (step // ACCUM_STEPS) * ACCUM_STEPS\n            group_size = min(ACCUM_STEPS, len(dl_tr)-group_start)\n            scaler.scale(loss/group_size).backward()\n            if (step+1) % ACCUM_STEPS == 0 or step+1 == len(dl_tr):\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)\n                old_scale = scaler.get_scale()\n                scaler.step(opt); scaler.update()\n                if scaler.get_scale() >= old_scale: sched.step()\n                opt.zero_grad(set_to_none=True)\n                if ema is not None:\n                    ema_steps += 1\n                    decay = min(EMA_DECAY, (1 + ema_steps) / (10 + ema_steps))\n                    with torch.no_grad():\n                        for pe, pm in zip(ema.parameters(), model.parameters()): pe.lerp_(pm, 1 - decay)\n                        for be, bm in zip(ema.buffers(), model.buffers()): be.copy_(bm)\n            total += loss.item()*len(x); count += len(x)\n        judged = ema if ema is not None else model\n        yv, pv = evaluate(judged, dl_val)\n        observed = np.isfinite(yv)\n        clipped = np.clip(pv, 1e-7, 1-1e-7)\n        val_loss = float((-(np.nan_to_num(yv)*np.log(clipped) +\n                           (1-np.nan_to_num(yv))*np.log1p(-clipped)))[observed].mean())\n        auc, _ = macro_auc(yv, pv, LABEL_COLS)\n        rec = dict(fold=fold, epoch=ep, loss=total/count, val_bce=val_loss,\n                   auc_pseudo=auc, minutes=(time.time()-t0)/60)\n        if REPORT_AUX_WEIGHT:\n            rec[\'report_aux\'] = aux_total / count\n        hist.append(rec); print(rec)\n        # Soft-label BCE on pseudo-labelled validation selects epochs; gold is never consulted.\n        if val_loss < best:\n            best = val_loss\n            atomic_torch_save(dict(model=judged.state_dict(), cfg=recipe, epoch=ep,\n                                   fold=fold, val_bce=best, auc_pseudo=auc), path)\n        atomic_torch_save(dict(model=model.state_dict(), cfg=recipe, epoch=ep, best=best,\n                               optimizer=opt.state_dict(), scheduler=sched.state_dict(),\n                               scaler=scaler.state_dict(), history=hist,\n                               ema=None if ema is None else ema.state_dict(), ema_steps=ema_steps), last_path)\n        pd.DataFrame(hist).to_csv(os.path.join(OUT, f\'history_fold_{fold}.csv\'), index=False)\n    model.load_state_dict(torch.load(path, map_location=DEV, weights_only=False)[\'model\'])\n    yv, pv = evaluate(model, dl_val)\n    output = pd.DataFrame(pv, columns=LABEL_COLS)\n    output.insert(0,\'StudyInstanceUID\',df.iloc[va_rows][\'StudyInstanceUID\'].to_numpy())\n    output[\'fold\'] = fold\n    output.to_csv(os.path.join(OUT,f\'oof_fold_{fold}.csv\'), index=False)\n    gold_rows = np.flatnonzero(folds == -1)\n    yg, pg = evaluate(model, make_loader(gold_rows, False, fold_seed))\n    gold_output = pd.DataFrame(pg, columns=LABEL_COLS)\n    gold_output.insert(0,\'StudyInstanceUID\',df.iloc[gold_rows][\'StudyInstanceUID\'].to_numpy())\n    gold_output.to_csv(os.path.join(OUT,f\'gold_fold_{fold}.csv\'), index=False)\n    print(\'Gold audit after checkpoint selection\'\n          + (\' (IN-SAMPLE: gold trained at weight %g)\' % GOLD_WEIGHT if GOLD_WEIGHT > 0 else \'\') + \':\',\n          macro_auc(yg,pg,LABEL_COLS)[0])\n    all_hist.extend(hist)\n    del model, ema, opt, sched, scaler, dl_tr, dl_val\n    gc.collect()\n    if DEV==\'cuda\': torch.cuda.empty_cache()\n\npaths = [os.path.join(OUT,f\'oof_fold_{f}.csv\') for f in range(N_FOLDS)]\nif all(os.path.exists(p) for p in paths):\n    oof = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)\n    expected = df.loc[folds>=0,\'StudyInstanceUID\']\n    assert oof[\'StudyInstanceUID\'].is_unique and set(oof[\'StudyInstanceUID\']) == set(expected)\n    aligned = oof.set_index(\'StudyInstanceUID\').loc[expected]\n    oof.to_csv(os.path.join(OUT,\'oof_predictions.csv\'), index=False)\n    print(\'OOF pseudo AUC (noisy-label proxy, selection-biased):\',\n          macro_auc(df.loc[folds>=0,LABEL_COLS].to_numpy(),aligned[LABEL_COLS].to_numpy(),LABEL_COLS))\nelse:\n    print(\'Partial fold run; complete all folds before comparing aggregate OOF results.\')\n\n# Fold predictions share the same gold study order; audit the exact inference blend.\ngold_paths = [os.path.join(OUT,f\'gold_fold_{f}.csv\') for f in range(N_FOLDS)]\nif all(os.path.exists(p) for p in gold_paths):\n    gold_ids = df.loc[folds==-1,\'StudyInstanceUID\']\n    if len(gold_ids):\n        members = [pd.read_csv(p).set_index(\'StudyInstanceUID\').loc[gold_ids,LABEL_COLS].to_numpy()\n                   for p in gold_paths]\n        blended = rank_blend(members)\n        audit = pd.DataFrame(blended,columns=LABEL_COLS)\n        audit.insert(0,\'StudyInstanceUID\',gold_ids.to_numpy())\n        audit.to_csv(os.path.join(OUT,\'gold_ensemble.csv\'),index=False)\n        print(\'Gold fold-ensemble audit\' + (\' (in-sample)\' if GOLD_WEIGHT > 0 else \'\') + \':\', macro_auc(df.loc[folds==-1,LABEL_COLS].to_numpy(),blended,LABEL_COLS))'
RAD_JOB_SOURCE = 'import os, gc, copy, math, time, json, random, warnings\nwarnings.filterwarnings("ignore")\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom tqdm.auto import tqdm\n\n# ---- inputs ------------------------------------------------------------------\n"""Bounded discovery of attached Kaggle artifacts; embedded in launch notebooks."""\nimport fnmatch, os\nfrom pathlib import Path\n\n\nDICOM_DIRS = (\'train_series\', \'test_series\')   # competition image trees: never listed\n\n\ndef attached_files(name, root=\'/kaggle/input\', max_depth=5):\n    # Kaggle mounts datasets and notebook outputs at different depths; artifacts live in\n    # folders 1-5 levels below the root. The DICOM trees are pruned: listing them took\n    # ~2 min on local disk and over an hour on the TPU VM\'s network-mounted inputs.\n    root = Path(root)\n    found = set()\n    for path, dirs, files in os.walk(root):\n        depth = len(Path(path).relative_to(root).parts)\n        dirs[:] = [d for d in dirs if d not in DICOM_DIRS] if depth < max_depth else []\n        if depth >= 1:\n            found.update(Path(path) / f for f in fnmatch.filter(files, name) if not f.startswith(\'.\'))\n    return sorted(found)\n\n\ndef one_input(name, override=\'\', root=\'/kaggle/input\'):\n    if override:\n        path = Path(override)\n        if not path.is_file():\n            raise FileNotFoundError(f\'Configured input does not exist: {path}\')\n        return str(path)\n    matches = attached_files(name, root)\n    if len(matches) != 1:\n        raise RuntimeError(f\'Attach exactly one intended {name} input or set its path explicitly. \'\n                           f\'Found: {[str(p) for p in matches]}\')\n    print(f\'{name}: {matches[0]}\')\n    return str(matches[0])\n\n\ndef fold_inputs(expected, root=\'/kaggle/input\', per_fold=1):\n    # per_fold > 1 when several training runs (e.g. DINOv2 and DINOv3 arms) are attached.\n    paths = []\n    for fold in expected:\n        matches = attached_files(f\'fold_{fold}.pt\', root)\n        if len(matches) != per_fold:\n            raise RuntimeError(f\'Attach exactly {per_fold} fold_{fold}.pt input(s) \'\n                               f\'({"exactly one" if per_fold == 1 else "one per training run"}). \'\n                               f\'Found: {[str(p) for p in matches]}\')\n        paths.extend(str(p) for p in matches)\n    for path in paths: print(f\'checkpoint: {path}\')\n    return paths\n\n\ndef checkpoint_inputs(root=\'/kaggle/input\'):\n    # Every attached fold_N.pt is an ensemble member; families are resolved from each\n    # checkpoint\'s recipe, and duplicates of one family-fold are rejected by the caller.\n    paths = [str(p) for p in attached_files(\'fold_*.pt\', root)]\n    if not paths:\n        raise RuntimeError(\'Attach at least one training output containing fold_N.pt\')\n    for path in paths: print(f\'checkpoint: {path}\')\n    return paths\n\n# Cache shards and label tables are discovered under /kaggle/input. Set CACHE_DIR to a\n# comma-separated list of shard folders only if the discovery is ambiguous.\nINPUT_ROOT = os.environ.get("KNEE_INPUT_ROOT", "/kaggle/input")   # other hosts: where inputs live\nCACHE_DIR = \',\'.join(str(p.parent) for p in attached_files(\'cache_meta.json\', INPUT_ROOT))\nassert CACHE_DIR, \'Attach the 02_cache shard outputs\'\nLABEL_ROOT = INPUT_ROOT\nCOMP_DIR  = os.environ.get("KNEE_COMP_DIR") or next(\n    (d for d in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection")\n     if os.path.isdir(d)), "/kaggle/input/competitions/rsna-knee-abnormality-detection")\nOUT       = os.environ.get("KNEE_OUT", "/kaggle/working")   # one folder per family in a session\n# -----------------------------------------------------------------------------\n# Several cache shards may be attached; a comma-separated list is accepted too.\nCACHE_DIRS = [d for d in CACHE_DIR.split(\',\') if d]\n\nLABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",\n              "PF OA","Effusion","Synovitis","Baker\'s","Contusion","Fracture"]\n\n_metas = [json.load(open(os.path.join(d, \'cache_meta.json\'))) for d in CACHE_DIRS]\n_recipe_only = lambda m: {k: v for k, v in m.items() if k not in (\'shard\', \'n_shards\')}\nassert all(_recipe_only(m) == _recipe_only(_metas[0]) for m in _metas), \'Cache shards come from different recipes\'\nCACHE_META = _recipe_only(_metas[0])\n_shards = sorted(m.get(\'shard\', 0) for m in _metas)\nassert _shards == list(range(_metas[0].get(\'n_shards\', 1))), f\'Attach every cache shard exactly once (found {_shards})\'\nassert CACHE_META.get(\'version\') in (\'knee-2c-v2\', \'knee-2c-v3\', \'knee-2c-v4\', \'knee-2c-v5\'), \'Rebuild the step-2c cache with current preprocessing\'\nPLANES = CACHE_META[\'slots\']\nCACHE_IMG = CACHE_META[\'img\']\nN_SLICES = CACHE_META[\'n_slices\']\n\n# ---- model recipe -------------------------------------------------------------\n# Eight self-trained families, chosen by PRESET (one launcher set per family):\n# reader_convnext: public 2.5D study reader (0.929 alone). ConvNeXt-T on windows of three\n#   adjacent slices, slot and stack-position tokens, two transformer layers, attention\n#   pooling per finding, EMA weights, warmup then cosine.\n# reader_student: the ConvNeXt reader trained on report labels mixed with our teachers\' OOF predictions.\n# reader_student2: round 2, whose teacher also includes the round-1 students.\n# reader_caformer: the same reader on CAFormer-S18 (ImageNet-22k), for backbone diversity.\n# reader_coatnet: the same reader on CoAtNet-RMLP-2 at 384 px, the backbone of the public\n#   Raptor single model (0.928 alone). Gradient checkpointing keeps it inside a T4.\n# slot_dinov2: the public DINOv2 member (one spread 3-slice image per slot, last six\n#   blocks fine-tuned, per-finding query head).\n# radimagenet: frozen official RadImageNet ResNet-50 slice tokens with a per-finding query\n#   head (public E9/E11 arm); encoded once, all folds in one run (rad_runtime.py).\n# kneenet_b0: the earlier EfficientNet-b0 model, kept for comparison.\nPRESET = os.environ.get("KNEE_PRESET", "reader_convnext")\nPRESETS = {\n    \'reader_convnext\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                            epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                            clip=3., grad_checkpoint=False),\n    # Noisy student: the same ConvNeXt reader recipe trained on targets mixed with the out-of-fold\n    # predictions of our own readers (teacher_targets.csv). Agreement-gated: the teacher share runs\n    # from 0.5 where the report readers agree confidently to 0.9 where they are split or silent.\n    # Slot dropout hides whole MRI sequences in training (test sites differ in protocol).\n    \'reader_student\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                           epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                           pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                           clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # Round 2: the same student recipe, whose teacher also averages the round-1 students\' OOF.\n    # 11 epochs: pseudo-validation plateaus by epochs 10-12, and a two-fold session then fits\n    # in under two GPU hours.\n    \'reader_student2\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                            epochs=11, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                            pct_start=1/11, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                            clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,\n                            teacher_families={\'reader_convnext\': 1.0, \'reader_caformer\': 1.0, \'reader_student\': 1.0}),\n    # Same reader, a different backbone family for ensemble diversity (convolution stages, then\n    # attention; ImageNet-22k). ~23% more activation memory per image than ConvNeXt-T, whose\n    # batch 4 peaked at 10.9 GB on a T4: half the micro-batch, same effective batch.\n    \'reader_caformer\': dict(arch=\'reader25d\', backbone=\'caformer_s18.sail_in22k_ft_in1k\', img=CACHE_IMG,\n                            epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                            clip=3., grad_checkpoint=False),\n    # Round-1 student recipe and teacher on a ResNet-50 (batch-norm CNN) for ensemble diversity:\n    # LB 2026-10-09/10: more ConvNeXt students/seeds added ~0.001; round-2 teachers did not help.\n    # Half the micro-batch (activation memory), same effective batch.\n    \'reader_student_r50\': dict(arch=\'reader25d\', backbone=\'resnet50.a1_in1k\', img=CACHE_IMG,\n                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # Report-grounded student: the round-1 student recipe and teacher plus an auxiliary loss that\n    # regresses each study\'s report embedding (scripts/report_embeddings.py; attach the\n    # knee-report-embeddings dataset) from the images, at this weight relative to the label BCE.\n    \'reader_student_rep\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                               epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,\n                               report_aux=.2),\n    # Round-1 student recipe and teacher on CAFormer-S18 (in22k): its readers (session J) beat the\n    # ConvNeXt readers on pseudo-validation (AUC 0.895/0.897 vs 0.893/0.891, folds 0/1); ResNet-50\n    # students fell far behind (0.915 vs 0.963), so diversity comes from this family instead.\n    \'reader_student_caf\': dict(arch=\'reader25d\', backbone=\'caformer_s18.sail_in22k_ft_in1k\', img=CACHE_IMG,\n                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # Round-1 student recipe and teacher on ConvNeXt-Base (ImageNet-22k), ~3x ConvNeXt-T compute:\n    # trained on the TPU (08_tpu_student_cnxb), where the long run does not hold up GPU sessions.\n    \'reader_student_cnxb\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # The round-1 ConvNeXt student recipe and teacher on the knee-2c-v4 cache (six v3 slots + two\n    # 90 mm local slots around the header-geometry joint anchor); a separate family name so its\n    # LB comparison with the v3 students (0.935 on folds 0-1) stays clean.\n    \'reader_student_v4\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                              epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # ConvNeXt-Base student on knee-2c-v4 (TPU): LB 2026-10-12 showed both gains separately,\n    # ConvNeXt-B v3 x5 = 0.938 vs ConvNeXt-T v3 x6 = 0.936, and v4 folds 0-1 = 0.938 vs v3 = 0.935.\n    \'reader_student_cnxb_v4\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                   epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                   pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                   clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # The ConvNeXt student on knee-2c-v5 (288 px, 10 windows per series; TPU).\n    # 288 px x 8 slots: half the T4 micro-batch of v4 (activation memory), same effective batch of 8.\n    \'reader_student_v5\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                              epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    # ConvNeXt-Base and CAFormer-S18 students on knee-2c-v5 (TPU / GPU after the 2026-10-17 quota reset).\n    # On a T4: one study per micro-batch (48 windows at 288 px) with gradient checkpointing; the TPU\n    # runtime ignores both (8 studies per step across the chips, no checkpointing). 8 epochs: at 50-60\n    # min per epoch on a T4, 14 would exceed the 12 h session (2026-10-14); ConvNeXt-B at 288 px\n    # reached val BCE 0.375 / AUC 0.954 by epoch 3.\n    \'reader_student_cnxb_v5\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                   epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                   pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                   clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2),\n    # v5 students taught partly by heliosli\'s compact model (its out-of-fold predictions on the training\n    # studies, 10_compact_oof_*): teacher = our ConvNeXt readers (sessions A-F, the Z5/B5 teacher; no CAFormer\n    # reader outputs are attached) and the compact model at 1.0 each, i.e. half ours and half compact, its\n    # per-finding ranking put on our teachers\' value distribution (its report targets use other label\n    # definitions, e.g. synovitis mean 0.54 vs 0.30);\n    # otherwise the ConvNeXt-T / -B v5 recipes.\n    \'reader_student_ct_v5\': dict(arch=\'reader25d\', backbone=\'convnext_tiny.in12k_ft_in1k\', img=CACHE_IMG,\n                                 epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                 pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                 clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,\n                                 teacher_families={\'reader_convnext\': 1.0, \'reader_compact\': 1.0},\n                                 teacher_quantile_map=(\'reader_compact\',)),\n    \'reader_student_cnxb_ct_v5\': dict(arch=\'reader25d\', backbone=\'convnext_base.fb_in22k_ft_in1k\', img=CACHE_IMG,\n                                      epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                      pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                      clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2,\n                                      teacher_families={\'reader_convnext\': 1.0, \'reader_compact\': 1.0},\n                                      teacher_quantile_map=(\'reader_compact\',)),\n    \'reader_student_caf_v5\': dict(arch=\'reader25d\', backbone=\'caformer_s18.sail_in22k_ft_in1k\', img=CACHE_IMG,\n                                  epochs=7, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                                  pct_start=1/7, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                                  clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),\n    \'reader_coatnet\': dict(arch=\'reader25d\', backbone=\'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\', img=384,\n                           epochs=10, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,\n                           pct_start=1/10, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,\n                           clip=3., grad_checkpoint=True),\n    \'slot_dinov2\': dict(arch=\'slot_query\', backbone=\'vit_small_patch14_dinov2.lvd142m\', img=224, epochs=10,\n                        batch=4, accum=2, lr=1e-3, encoder_lr_scale=8e-3, wd=.02, pct_start=.15, dropout=.2,\n                        unfreeze_last=6, head_hidden=256, ema=0., clip=5., grad_checkpoint=False),\n    \'radimagenet\': dict(arch=\'radhead\', backbone=\'radimagenet_resnet50\', img=224, epochs=24, batch=48,\n                        accum=1, lr=2e-4, encoder_lr_scale=0., wd=3e-3, pct_start=0., dropout=.15,\n                        unfreeze_last=None, head_hidden=512, ema=0., clip=1., grad_checkpoint=False),\n    \'kneenet_b0\': dict(arch=\'kneenet\', backbone=\'tf_efficientnet_b0.ns_jft_in1k\', img=CACHE_IMG, epochs=6,\n                       batch=4, accum=1, lr=3e-4, encoder_lr_scale=.3, wd=1e-2, pct_start=.1, dropout=.4,\n                       unfreeze_last=None, head_hidden=None, ema=0., clip=5., grad_checkpoint=False),\n}\n_preset = PRESETS[PRESET]\n_needs = {\'_v4\': \'knee-2c-v4\', \'_v5\': \'knee-2c-v5\'}.get(PRESET[-3:])\nassert _needs is None or CACHE_META.get(\'version\') == _needs, f\'Attach the {_needs} cache outputs\'\nARCH = _preset[\'arch\']\nBACKBONE = os.environ.get("KNEE_BACKBONE", _preset[\'backbone\'])\nIMG = _preset[\'img\']              # model input size; the cache is resized on the GPU if different\nEPOCHS = _preset[\'epochs\']\nBATCH = _preset[\'batch\']          # studies per micro-batch\nACCUM_STEPS = _preset[\'accum\']    # effective batch = BATCH * ACCUM_STEPS studies\nLR = _preset[\'lr\']                # head learning rate\nENCODER_LR_SCALE = _preset[\'encoder_lr_scale\']   # encoder rate = LR * this\nWD = _preset[\'wd\']\nPCT_START = _preset[\'pct_start\']  # warmup share of the one-cycle schedule\nDROPOUT = _preset[\'dropout\']\nUNFREEZE_LAST = _preset[\'unfreeze_last\']\nHEAD_HIDDEN = _preset[\'head_hidden\']\nEMA_DECAY = _preset[\'ema\']        # 0 disables; EMA weights are validated and saved\nCLIP = _preset[\'clip\']\nGRAD_CHECKPOINT = _preset[\'grad_checkpoint\']\nTEACHER_WEIGHT = _preset.get(\'teacher\', 0.0)   # constant or (lo, hi) gate; attach teacher_targets.csv\nSLOT_DROPOUT = _preset.get(\'slot_dropout\', 0.0) # reader25d: chance of hiding each present series in training\nTEACHER_FAMILY_WEIGHTS = _preset.get(\'teacher_families\')   # None: label_builder.TEACHER_FAMILIES\nTEACHER_QUANTILE_MAP = _preset.get(\'teacher_quantile_map\', ())   # outside teachers put on our teachers\' scale\nREPORT_AUX = _preset.get(\'report_aux\', 0.0)    # weight of the report-embedding auxiliary loss; 0 disables\nTRAIN_WINDOWS = 6                 # reader25d: windows encoded per series per step (of 8)\n# Gold studies train in every fold at this per-cell weight (decision 2026-10-07). Their\n# audit AUC is then in-sample and is printed as such; selection never reads gold.\nGOLD_WEIGHT = 3.0\nREADER_WEIGHTS = {}               # e.g. {\'extra_labels_sol_v2.csv\': 2.0}; default 1 each\n# Every reader the run must find; a missing attachment stops before any GPU time is spent.\nREQUIRED_READERS = [\'report_labels_v2.csv\', \'llm_labels_v4_blend.csv\', \'labels_llm_gpt56sol.csv\',\n                    \'qwen_knee_weak_labels.csv\', \'extra_labels_sol_v2.csv\',\n                    \'extra_labels_sonnet_legacy.csv\', \'pseudo_google_gemini*flash*.jsonl\']\nNORMALIZATION = \'pretrained\'\nN_FOLDS = 5\nFOLDS_TO_RUN = json.loads(os.environ.get("KNEE_FOLDS", "[0,1,2,3,4]"))  # one fold first on Kaggle\nGROUP_COL = \'report_group\'        # folds keep duplicate reports together\nRESUME = False  # copy both fold_N.pt and last_fold_N.pt to OUT before enabling\nT_TRAIN     = 10       # kneenet only: contiguous slices sampled per plane per step\nSEED        = int(os.environ.get("KNEE_SEED", 42))   # sessions may train extra seeds of a family\nNUM_WORKERS = int(os.environ.get("KNEE_WORKERS", 4))   # Kaggle gives 4 cores; two-GPU sessions use 2 per job\n\ndef write_atomic(path, content):\n    # Two jobs in one session may write the same small file (folds, config); replace it whole.\n    # content: text, or a function that writes to the temporary path it is given.\n    temp = f\'{path}.{os.getpid()}.tmp\'\n    if callable(content):\n        content(temp)\n    else:\n        with open(temp, \'w\') as handle: handle.write(content)\n    os.replace(temp, path)\n\n\nos.makedirs(OUT, exist_ok=True)\nDEV = "cuda" if torch.cuda.is_available() else "cpu"\ntorch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)\ntorch.backends.cudnn.benchmark = True          # shapes are fixed -> let cuDNN autotune\n\nprint("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")\nif DEV == "cuda" and torch.cuda.device_count() > 1:\n    print(f"note: {torch.cuda.device_count()} GPUs visible, using 1 by design "\n          f"(DataParallel benchmarked 13x slower here)")\nif ARCH == \'kneenet\' and BACKBONE.startswith(\'vit_\'):\n    BATCH, ACCUM_STEPS = 1, 4  # effective batch 4; lower activation memory\nprint(f"recipe: {PRESET} ({ARCH}) | {BACKBONE} @ {IMG}px | {CACHE_META[\'version\']} {len(PLANES)} slots x "\n      f"{N_SLICES} slices x {CACHE_IMG}px in {len(CACHE_DIRS)} shard(s) | {EPOCHS} epochs, "\n      f"{BATCH}x{ACCUM_STEPS} studies/step | gold weight {GOLD_WEIGHT}")\nassert DEV == "cuda" or os.environ.get("KNEE_ALLOW_CPU"), "Enable a GPU before starting training"\n\nRESUME_DIR = os.environ.get("KNEE_RESUME_DIR", "")\nif RESUME_DIR:\n    import shutil\n    for fold in FOLDS_TO_RUN:\n        for filename in (f\'fold_{fold}.pt\', f\'last_fold_{fold}.pt\'):\n            previous = Path(RESUME_DIR) / os.environ.get(\'KNEE_RUN_NAME\', PRESET) / filename\n            if not previous.is_file():\n                continue                       # this fold has no earlier progress\n            destination = Path(OUT) / filename\n            assert not destination.exists(), f\'Restart the session before copying {destination}\'\n            shutil.copy2(previous, destination)\n            RESUME = True\n\n\n"""Soft training targets from report-label readers; embedded in training.ipynb."""\nimport os, re, json, fnmatch, hashlib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\n# Public report-label tables (rules allow public external data). File name -> suffix of\n# the twelve target columns. The mean of these four reached 0.896 macro AUC on the gold\n# studies in the public 2.5D reader\'s own measurement.\nPUBLIC_LABEL_TABLES = {\n    \'report_labels_v2.csv\': \'\',            # pilkwang/rsna-knee-llm-labels\n    \'llm_labels_v4_blend.csv\': \'\',         # stevenleehans/rsna-knee-llm-report-labels\n    \'labels_llm_gpt56sol.csv\': \'\',         # lixin73/rsna-knee-llm-report-labels-sol56\n    \'qwen_knee_weak_labels.csv\': \'__label\',  # laymond/rsna-knee-abnormality-qwen3-8b-weak-labels\n}\n# Our own readers:\n#  - extra_labels_*.csv from scripts/export_reader_labels.py (GPT Sol v2, Sonnet legacy);\n#  - the Flash member file written by our 01_labels notebook (its labels/ folder), read\n#    directly so Flash counts once. That notebook\'s consensus train_targets.csv is not read:\n#    it already mixes Flash with Sonnet.\nOWN_PREFIX = \'extra_labels\'\nOWN_JSONL_READERS = {\'pseudo_google_gemini*flash*.jsonl\': \'legacy\'}\n\n# State -> probability that the finding is positive. Legacy is the 01_labels notebook\'s own\n# HARD map; v2 states follow the knee-report-labels-v2 protocol. Unsure abstains (NaN)\n# under legacy; under v2 it marks an affirmative finding of undecidable eligibility.\nSTATE_P = {\n    \'legacy\': {\'definite\': 1.0, \'probable\': .75, \'possible\': .35, \'absent\': 0.0},\n    \'v2\': {\'definite\': 1.0, \'unsure\': .5, \'absent\': 0.0},\n}\nSTATE_KEYS = dict(zip(\n    ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n     "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"],\n    ["acl", "mcl", "medial_meniscus", "lateral_meniscus", "medial_oa", "lateral_oa",\n     "pf_oa", "effusion", "synovitis", "bakers", "contusion", "fracture"]))\n\n\ndef iter_jsonl(path):\n    # One historical shard was written with literal "\\n" separators; split on both.\n    with open(path) as handle:\n        for chunk in re.split(r\'\\n|\\\\n\', handle.read()):\n            if chunk.strip():\n                yield json.loads(chunk)\n\n\ndef states_frame(records, label_cols, protocol):\n    mapping = STATE_P[protocol]\n    rows = [{\'StudyInstanceUID\': str(r[\'StudyInstanceUID\']),\n             **{c: mapping.get(str(r.get(STATE_KEYS.get(c, c), \'\')).lower()) for c in label_cols}}\n            for r in records]\n    frame = pd.DataFrame(rows, columns=[\'StudyInstanceUID\', *label_cols])\n    return frame.drop_duplicates(\'StudyInstanceUID\', keep=\'last\')\n\n\ndef find_label_tables(root=\'/kaggle/input\'):\n    found = {}\n    for path, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in (\'train_series\', \'test_series\')]\n        for name in files:\n            if (name in PUBLIC_LABEL_TABLES or (name.startswith(OWN_PREFIX) and name.endswith(\'.csv\'))\n                    or any(fnmatch.fnmatch(name, pattern) for pattern in OWN_JSONL_READERS)):\n                found.setdefault(name, []).append(os.path.join(path, name))\n    duplicates = {k: v for k, v in found.items() if len(v) > 1}\n    assert not duplicates, f\'Attach each label table once: {duplicates}\'\n    flash = [k for k in found if any(fnmatch.fnmatch(k, p) for p in OWN_JSONL_READERS)]\n    assert len(flash) <= 1, f\'More than one Flash member file attached: {flash}\'\n    return {k: v[0] for k, v in sorted(found.items())}\n\n\ndef read_reader(path, label_cols):\n    name = os.path.basename(path)\n    if name.endswith(\'.jsonl\'):\n        protocol = next(v for p, v in OWN_JSONL_READERS.items() if fnmatch.fnmatch(name, p))\n        frame = states_frame(iter_jsonl(path), label_cols, protocol)\n        columns = list(label_cols)\n    else:\n        suffix = PUBLIC_LABEL_TABLES.get(name, \'\')\n        frame = pd.read_csv(path, dtype={\'StudyInstanceUID\': str})\n        columns = [c + suffix for c in label_cols]\n    missing = [c for c in columns if c not in frame.columns]\n    assert not missing, f\'{name} lacks {missing[:3]}\'\n    assert not frame[\'StudyInstanceUID\'].duplicated().any(), f\'{name} repeats studies\'\n    values = frame.set_index(\'StudyInstanceUID\')[columns].apply(pd.to_numeric, errors=\'coerce\')\n    values.columns = label_cols\n    bad = (values < 0) | (values > 1)\n    if bad.any().any():\n        print(f\'{name}: {int(bad.sum().sum())} values outside 0..1 treated as unobserved\')\n        values = values.mask(bad)\n    return values\n\n\n# Noisy-student targets (2026-10-08). teacher_targets.csv holds out-of-fold predictions of our own\n# image models (scripts/teacher_targets.py); image models average out report noise, so their OOF\n# predictions are cleaner targets than any single report reader. Gold rows never take teacher values.\nTEACHER_FILE = \'teacher_targets.csv\'\n\n\ndef find_teacher(root=\'/kaggle/input\'):\n    hits = []\n    for path, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in (\'train_series\', \'test_series\')]\n        if TEACHER_FILE in files:\n            hits.append(os.path.join(path, TEACHER_FILE))\n    assert len(hits) <= 1, f\'Attach one teacher table, found {hits}\'\n    return hits[0] if hits else None\n\n\n# Family -> weight for teachers. Readers only: on the public LB the slot and RadImageNet families\n# lowered the blend (17 models at equal weights 0.917 < two reader folds 0.923). Students and\n# third-party checkpoints are never teachers.\nTEACHER_FAMILIES = {\'reader_convnext\': 1.0, \'reader_caformer\': 1.0}\n\n\ndef build_teacher(root, label_cols, gold_ids=(), families=None, quantile_map=()):\n    """Teacher probabilities from the out-of-fold predictions (oof_fold_N.csv) of our own models\n    found under root: seeds of a family are averaged, then families by weight. Every value is\n    out-of-fold. Families in quantile_map (an outside model trained on other label definitions)\n    keep their per-finding ranking but take the value distribution of the weighted mean of the\n    other families before the average. Returns (frame with StudyInstanceUID, label_cols, n_members; meta)."""\n    families = TEACHER_FAMILIES if families is None else families\n    family_of = lambda folder: re.sub(r\'_s\\d+$\', \'\', folder)\n    members = {}\n    for path, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in (\'train_series\', \'test_series\')]\n        folder = os.path.basename(path)\n        oofs = sorted(f for f in files if re.fullmatch(r\'oof_fold_\\d+\\.csv\', f))\n        if oofs and family_of(folder) in families:\n            members.setdefault(folder, []).extend(\n                pd.read_csv(os.path.join(path, f), dtype={\'StudyInstanceUID\': str}) for f in oofs)\n    assert members, f\'no teacher OOF predictions under {root}\'\n    gold_ids, per_family = set(gold_ids), {}\n    meta = dict(members={}, families={}, weights=dict(families))\n    for folder, parts in sorted(members.items()):\n        frame = pd.concat(parts, ignore_index=True)\n        assert frame[\'StudyInstanceUID\'].is_unique, f\'{folder}: a study is out-of-fold twice\'\n        assert not gold_ids & set(frame[\'StudyInstanceUID\']), f\'{folder}: gold study in OOF\'\n        meta[\'members\'][folder] = dict(studies=len(frame), folds=sorted(int(f) for f in frame[\'fold\'].unique()))\n        values = frame.set_index(\'StudyInstanceUID\')[label_cols].astype(float)\n        assert ((values >= 0) & (values <= 1)).all().all(), f\'{folder}: OOF outside 0..1\'\n        per_family.setdefault(family_of(folder), []).append(values)\n    ids = sorted(set().union(*[f.index for frames in per_family.values() for f in frames]))\n    num = np.zeros((len(ids), len(label_cols))); den = np.zeros_like(num)\n    n_members = np.zeros(len(ids), int)\n    means = {}\n    for family, frames in per_family.items():\n        stack = np.stack([f.reindex(ids).to_numpy(float) for f in frames])\n        counts = np.isfinite(stack).sum(axis=0)\n        means[family] = (np.where(counts > 0, np.nansum(stack, axis=0) / np.maximum(counts, 1), np.nan), stack)\n    mapped = [f for f in quantile_map if f in means]\n    if mapped:\n        ref_num = sum(np.nan_to_num(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)\n        ref_den = sum(np.isfinite(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)\n        assert np.max(ref_den) > 0, \'quantile_map needs at least one other teacher family\'\n        ref = np.where(ref_den > 0, ref_num / np.maximum(ref_den, 1e-9), np.nan)\n        for family in mapped:\n            m = means[family][0].copy()\n            for j in range(m.shape[1]):\n                ok, r = np.isfinite(m[:, j]), np.isfinite(ref[:, j])\n                if ok.any() and r.any():\n                    pct = (pd.Series(m[ok, j]).rank(method=\'average\').to_numpy() - .5) / ok.sum()\n                    m[ok, j] = np.quantile(ref[r, j], pct)\n            means[family] = (m, means[family][1])\n        meta[\'quantile_mapped\'] = mapped\n    for family, (mean, stack) in means.items():\n        ok = np.isfinite(mean)\n        num += np.where(ok, mean, 0) * families[family]\n        den += ok * families[family]\n        n_members += np.isfinite(stack).all(axis=2).sum(axis=0)\n        meta[\'families\'][family] = dict(members=len(frames), studies=int(ok.all(axis=1).sum()))\n    teacher = pd.DataFrame(np.where(den > 0, num / np.maximum(den, 1e-9), np.nan), columns=label_cols)\n    teacher.insert(0, \'StudyInstanceUID\', ids)\n    teacher[\'n_members\'] = n_members\n    return teacher, meta\n\n\ndef apply_teacher(frame, teacher_path, label_cols, weight):\n    """Mix teacher probabilities into the pseudo rows: y = (1 - a) * report + a * teacher.\n    weight is a constant a, or a gate (lo, hi): a = lo + (hi - lo) * u with u = 1 - |2 * report - 1|,\n    so cells where the report readers agree confidently (consensus near 0 or 1) keep more of the\n    report and split or uncertain cells lean on the image teacher. A pseudo cell without a report\n    label takes the teacher value at cell weight hi. Gold rows are returned unchanged.\n    Returns (frame, summary)."""\n    lo, hi = (weight, weight) if np.isscalar(weight) else tuple(weight)\n    assert 0 <= lo <= hi <= 1 and hi > 0\n    teacher = pd.read_csv(teacher_path, dtype={\'StudyInstanceUID\': str})\n    assert teacher[\'StudyInstanceUID\'].is_unique, \'teacher repeats studies\'\n    t = teacher.set_index(\'StudyInstanceUID\')[label_cols].apply(pd.to_numeric, errors=\'coerce\')\n    assert (t.isna() | ((t >= 0) & (t <= 1))).all().all(), \'teacher values outside 0..1\'\n    out = frame.copy()\n    rows = np.flatnonzero(out[\'source\'].eq(\'pseudo\').to_numpy())\n    wcols = [f\'w_{c}\' for c in label_cols]\n    y = np.array(out.loc[out.index[rows], label_cols], dtype=float)\n    w = np.array(out.loc[out.index[rows], wcols], dtype=float)\n    tv = np.array(t.reindex(out[\'StudyInstanceUID\'].iloc[rows]), dtype=float)\n    has_t = np.isfinite(tv)\n    both, only_t = has_t & np.isfinite(y), has_t & ~np.isfinite(y)\n    a = lo + (hi - lo) * (1 - np.abs(2 * np.where(np.isfinite(y), y, .5) - 1))\n    y[both] = (1 - a[both]) * y[both] + a[both] * tv[both]\n    y[only_t] = tv[only_t]\n    w[only_t] = hi\n    out.loc[out.index[rows], label_cols] = np.where(np.isfinite(y), np.clip(y, .01, .99), np.nan).astype(np.float32)\n    out.loc[out.index[rows], wcols] = w.astype(np.float32)\n    summary = dict(teacher=teacher_path, weight=[lo, hi], mean_teacher_share=float(a[both].mean()) if both.any() else 0.,\n                   studies=int(has_t.any(axis=1).sum()),\n                   pseudo_studies=len(rows), mixed_cells=int(both.sum()), filled_cells=int(only_t.sum()),\n                   sha256=hashlib.sha256(open(teacher_path, \'rb\').read()).hexdigest())\n    return out, summary\n\n\ndef report_groups(reports):\n    # Byte-identical (up to case/punctuation) reports share targets: keep them in one fold.\n    norm = reports.fillna(\'\').astype(str).str.lower().map(lambda t: re.sub(r\'\\W+\', \'\', t))\n    return norm.map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:16]).to_numpy()\n\n\ndef build_targets(train, label_cols, tables, reader_weights=None, gold_weight=3.0):\n    """train: competition train.csv. Returns (frame, report) where frame holds soft\n    targets, per-cell weights (w_<col>), source and report group per study."""\n    ids = train[\'StudyInstanceUID\'].astype(str)\n    readers = {name: read_reader(path, label_cols).reindex(ids) for name, path in tables.items()}\n    assert readers, \'Attach at least one report-label table\'\n    weights = {name: (reader_weights or {}).get(name, 1.0) for name in readers}\n    num = sum(r[label_cols].fillna(0).to_numpy() * weights[n] for n, r in readers.items())\n    den = sum(r[label_cols].notna().to_numpy() * weights[n] for n, r in readers.items())\n    with np.errstate(invalid=\'ignore\', divide=\'ignore\'):\n        soft = np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)\n    soft = np.clip(soft, .01, .99)\n    gold = train[label_cols].apply(pd.to_numeric, errors=\'coerce\')\n    is_gold = gold.notna().all(axis=1).to_numpy()\n    y = soft.astype(np.float32)\n    w = np.where(np.isfinite(soft), 1.0, 0.0).astype(np.float32)\n    y[is_gold] = gold.to_numpy(np.float32)[is_gold]\n    w[is_gold] = gold_weight\n    frame = pd.DataFrame(y, columns=label_cols)\n    frame.insert(0, \'StudyInstanceUID\', ids.to_numpy())\n    for k, c in enumerate(label_cols): frame[f\'w_{c}\'] = w[:, k]\n    frame[\'source\'] = np.where(is_gold, \'gold\', \'pseudo\')\n    frame[\'report_group\'] = report_groups(train[\'Report\']) if \'Report\' in train else ids.to_numpy()\n    # Diagnostic only: each reader against the image-read gold labels. Not used for any choice.\n    report = {}\n    for name, r in {**readers, \'consensus\': pd.DataFrame(soft, index=ids, columns=label_cols)}.items():\n        aucs = []\n        # A reader that never labelled the gold studies has no gold AUC (not 0.5).\n        gold_rows = int(r.loc[ids[is_gold], label_cols].notna().any(axis=1).sum())\n        for c in label_cols:\n            truth, score = gold.loc[is_gold, c].to_numpy(), r.loc[ids[is_gold], c].fillna(.5).to_numpy()\n            aucs.append(roc_auc_score(truth, score) if len(np.unique(truth)) == 2 and gold_rows else np.nan)\n        finite = [a for a in aucs if np.isfinite(a)]\n        report[name] = dict(gold_macro_auc=float(np.mean(finite)) if finite else float(\'nan\'),\n                            gold_rows=gold_rows, weight=weights.get(name, 0),\n                            coverage=float(np.isfinite(r.to_numpy()).mean()))\n    return frame, report\n\n# ---- cache shards -------------------------------------------------------------\n# Rows are addressed as (shard, row in shard); every shard shares one recipe.\nparts = []\nfor shard_id, directory in enumerate(CACHE_DIRS):\n    part = pd.read_csv(os.path.join(directory, "cache_index.csv"))\n    shape = np.load(os.path.join(directory, "cache.npy"), mmap_mode="r").shape\n    assert shape[1:] == (len(PLANES), N_SLICES, CACHE_IMG, CACHE_IMG), f\'{directory}: cache shape {shape}\'\n    assert len(part) == shape[0] and part[\'row\'].between(0, shape[0] - 1).all()\n    parts.append(part.assign(cache_id=shard_id))\n    print(f"cache shard {shard_id}: {shape[0]:,} studies, {np.prod(shape)/1e9:.1f} GB  ({directory})")\nindex = pd.concat(parts, ignore_index=True)\nassert index[\'StudyInstanceUID\'].is_unique, \'a study appears in two cache shards\'\nT_TRAIN = min(T_TRAIN, N_SLICES)\n\n# ---- targets: report-label readers + gold ------------------------------------\ntrain_csv = pd.read_csv(os.path.join(COMP_DIR, \'train.csv\'), dtype={\'StudyInstanceUID\': str})\nLABEL_TABLES = find_label_tables(LABEL_ROOT)\nprint(\'label readers:\', list(LABEL_TABLES))\n_missing = [r for r in REQUIRED_READERS if not any(fnmatch.fnmatch(name, r) for name in LABEL_TABLES)]\nassert not _missing, f\'Attach the label inputs providing {_missing} (see KAGGLE_RUN/README.md)\'\ntargets, label_report = build_targets(train_csv, LABEL_COLS, LABEL_TABLES, READER_WEIGHTS, GOLD_WEIGHT)\nprint(pd.DataFrame(label_report).T.round(3).to_string())\nprint(\'(gold AUC of each reader is a diagnostic only; it chooses nothing. NaN: the reader has no gold rows)\')\nTEACHER_SUMMARY = None\nif np.max(TEACHER_WEIGHT) > 0:\n    _teacher = find_teacher(LABEL_ROOT)\n    if _teacher is None:\n        # No teacher table attached: build it here from the attached reader session outputs\n        # (their out-of-fold predictions), exactly as 07_teacher_targets would.\n        _gold = set(targets.loc[targets[\'source\'].eq(\'gold\'), \'StudyInstanceUID\'])\n        _frame, _meta = build_teacher(LABEL_ROOT, LABEL_COLS, gold_ids=_gold, families=globals().get(\'TEACHER_FAMILY_WEIGHTS\'),\n                                      quantile_map=globals().get(\'TEACHER_QUANTILE_MAP\', ()))\n        print(\'teacher built in session from:\', json.dumps(_meta[\'members\']))\n        _absent = sorted(f for f, w in (globals().get(\'TEACHER_FAMILY_WEIGHTS\') or {}).items() if w > 0 and f not in _meta[\'families\'])\n        assert not _absent, f\'Attach the teacher inputs for families {_absent}\'\n        os.makedirs(OUT, exist_ok=True)\n        _teacher = os.path.join(OUT, \'teacher_targets_used.csv\')\n        write_atomic(_teacher, lambda p: _frame.to_csv(p, index=False))\n    targets, TEACHER_SUMMARY = apply_teacher(targets, _teacher, LABEL_COLS, TEACHER_WEIGHT)\n    print(\'teacher targets:\', TEACHER_SUMMARY)\nwrite_atomic(os.path.join(OUT, \'targets_used.csv\'), lambda p: targets.to_csv(p, index=False))\n\nWEIGHT_COLS = [f\'w_{c}\' for c in LABEL_COLS]\ndf = index[index["n_present"] > 0].merge(targets, on="StudyInstanceUID", how="inner")\ndf = df[df[LABEL_COLS].notna().any(axis=1)].reset_index(drop=True)\nprint(f"\\nusable studies: {len(df):,}  ({df[\'source\'].value_counts().to_dict()})")\n# Report-embedding auxiliary targets (training only): NaN rows have no embedding and are masked.\nREP_COLS, AUX_DIM, REPORT_AUX_SUMMARY = [], 0, None\nif globals().get(\'REPORT_AUX\', 0.0) > 0:\n    _rep_path = one_input(\'report_embeddings.csv\', root=LABEL_ROOT)\n    _rep = pd.read_csv(_rep_path, dtype={\'StudyInstanceUID\': str})\n    REP_COLS = [c for c in _rep.columns if c.startswith(\'rep_\')]\n    assert REP_COLS and _rep[\'StudyInstanceUID\'].is_unique and not set(REP_COLS) & set(df.columns)\n    df = df.merge(_rep[[\'StudyInstanceUID\', *REP_COLS]], on=\'StudyInstanceUID\', how=\'left\')\n    AUX_DIM = len(REP_COLS)\n    import hashlib as _hashlib\n    REPORT_AUX_SUMMARY = dict(weight=REPORT_AUX, dims=AUX_DIM,\n                              studies=int(df[REP_COLS[0]].notna().sum()),\n                              sha256=_hashlib.sha256(open(_rep_path, \'rb\').read()).hexdigest())\n    print(\'report auxiliary targets:\', REPORT_AUX_SUMMARY)\ny_all = df[LABEL_COLS].to_numpy(np.float32)\nprint(f"observed label cells: {(~np.isnan(y_all)).mean():.1%}")\nprint("\\nmean soft target (observed cells):")\nprint(pd.Series(np.nanmean(y_all[df[\'source\'].eq(\'pseudo\')], axis=0), index=LABEL_COLS).round(3).to_string())\nassert df[\'source\'].isin([\'gold\',\'pseudo\']).all(), \'Unknown label source\'\nassert ((np.isnan(y_all)) | ((y_all>=0)&(y_all<=1))).all(), \'Targets outside [0,1]\'\ngold_check = train_csv.set_index(\'StudyInstanceUID\')[LABEL_COLS].apply(pd.to_numeric, errors=\'coerce\')\ngold_ids = df.loc[df[\'source\'].eq(\'gold\'), \'StudyInstanceUID\']\nassert np.array_equal(df.loc[df[\'source\'].eq(\'gold\'), LABEL_COLS].to_numpy(float),\n                      gold_check.loc[gold_ids].to_numpy(float)), \'Gold rows must carry the official labels\'\n\n"""Fold creation and metrics with explicit gold exclusion."""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\n\ndef assign_folds(df, label_cols, n_splits=5, seed=42, group_col=None):\n    folds = np.full(len(df), -1, dtype=int)\n    pseudo = np.flatnonzero(df[\'source\'].eq(\'pseudo\').to_numpy())\n    assert len(pseudo) >= n_splits\n    y = df.iloc[pseudo][label_cols].to_numpy(float)\n    # Balance missingness as well as positive findings. No gold target enters the split.\n    strata = np.concatenate([np.nan_to_num(y, nan=0) >= .5, np.isfinite(y)], axis=1).astype(int)\n    groups = df.iloc[pseudo][group_col].astype(str).to_numpy() if group_col else df.iloc[pseudo][\'StudyInstanceUID\'].to_numpy()\n    if len(np.unique(groups)) < len(groups):\n        splitter = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        splits = splitter.split(pseudo, groups=groups)\n    else:\n        splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        splits = splitter.split(pseudo, strata)\n    for fold, (_, val) in enumerate(splits): folds[pseudo[val]] = fold\n    assert (folds[pseudo] >= 0).all()\n    assert (folds[df[\'source\'].eq(\'gold\')] == -1).all()\n    return folds\n\n\ndef macro_auc(y, scores, label_cols):\n    aucs = {}\n    for k, name in enumerate(label_cols):\n        ok = np.isfinite(y[:, k]) & np.isfinite(scores[:, k])\n        binary = (y[ok, k] >= .5).astype(int)\n        aucs[name] = roc_auc_score(binary, scores[ok, k]) if len(np.unique(binary)) == 2 else np.nan\n    valid = [v for v in aucs.values() if np.isfinite(v)]\n    return (float(np.mean(valid)) if valid else float(\'nan\')), aucs\n\n\ndef rank_blend(predictions):\n    arrays = [np.asarray(p, dtype=float) for p in predictions]\n    assert arrays and all(a.shape == arrays[0].shape for a in arrays)\n    assert all(np.isfinite(a).all() for a in arrays)\n    return np.mean([pd.DataFrame(a).rank(pct=True, method=\'average\').to_numpy() for a in arrays], axis=0)\n\n"""Training dataset, embedded in training.ipynb after the fold helpers."""\nimport cv2\n\nclass KneeDS(Dataset):\n    def __init__(self, df, rows, train):\n        self.df = df.reset_index(drop=True)\n        self.rows = rows\n        self.train = train\n        self.y = self.df[LABEL_COLS].to_numpy(np.float32)\n        # Per-cell loss weights (gold rows are up-weighted); unweighted tables default to 1.\n        weight_cols = [f\'w_{c}\' for c in LABEL_COLS]\n        self.w = (self.df[weight_cols].to_numpy(np.float32) if set(weight_cols) <= set(self.df)\n                  else np.ones_like(self.y))\n        self.present = self.df[[f"present_{s}" for s in range(len(PLANES))]].to_numpy(bool)\n        self.row_of = self.df["row"].to_numpy(int)\n        self.shard_of = (self.df["cache_id"].to_numpy(int) if "cache_id" in self.df\n                         else np.zeros(len(self.df), int))\n        self.caches = {}                       # memmaps opened lazily, per worker and shard\n        # Training batches carry the report-embedding targets after the labels (weight 1 where known).\n        rep = globals().get(\'REP_COLS\') or []\n        self.aux = self.df[rep].to_numpy(np.float32) if (train and rep) else None\n\n    def _cache(self, shard):\n        if shard not in self.caches:\n            self.caches[shard] = np.load(os.path.join(CACHE_DIRS[shard], "cache.npy"), mmap_mode="r")\n        return self.caches[shard]\n\n    def __len__(self):\n        return len(self.rows)\n\n    def _aug(self, vol):\n        # NO horizontal or vertical flip: a horizontal flip in the coronal plane swaps\n        # medial and lateral, which would invert four of the twelve labels.\n        h, w = vol.shape[-2:]\n        ang = np.random.uniform(-12, 12)\n        sc  = np.random.uniform(0.9, 1.1)\n        M = cv2.getRotationMatrix2D((w/2, h/2), ang, sc)\n        M[:, 2] += np.random.uniform(-0.06, 0.06, 2) * w\n        out = np.empty_like(vol)\n        for i in range(vol.shape[0]):\n            out[i] = cv2.warpAffine(vol[i], M, (w, h), flags=cv2.INTER_LINEAR,\n                                    borderValue=0)\n        if np.random.rand() < 0.5:\n            out = np.clip(out.astype(np.float32) * np.random.uniform(0.85, 1.15)\n                          + np.random.uniform(-15, 15), 0, 255).astype(np.uint8)\n        return out\n\n    def _aug_slot(self, vol):\n        # Public-member jitter for slot_query: small rotation, zoom-in only (zooming out\n        # would invent tissue at the crop edge, where the popliteal fossa sits), edge\n        # replication, multiplicative intensity. Still no flips.\n        h, w = vol.shape[-2:]\n        M = cv2.getRotationMatrix2D((w/2, h/2), np.random.uniform(-8, 8),\n                                    np.random.uniform(1.0, 1.08))\n        M[:, 2] += np.random.uniform(-0.025, 0.025, 2) * w\n        out = np.stack([cv2.warpAffine(s, M, (w, h), flags=cv2.INTER_LINEAR,\n                                       borderMode=cv2.BORDER_REPLICATE) for s in vol])\n        return np.clip(out.astype(np.float32) * np.random.uniform(0.9, 1.1),\n                       0, 255).astype(np.uint8)\n\n    def __getitem__(self, i):\n        j = self.rows[i]\n        r = self.row_of[j]\n        cache = self._cache(self.shard_of[j])\n        present = self.present[j]\n\n        if self.train and ARCH == \'reader25d\':\n            # All windows; the model subsamples windows and augments each series on GPU.\n            vol = np.asarray(cache[r])\n        elif self.train and ARCH == \'slot_query\':\n            # One slice from each third of the band, drawn independently per slot, so\n            # every cached position trains while each input still spans the joint.\n            parts = slice_thirds(cache.shape[2])\n            vol = np.stack([self._aug_slot(np.asarray(cache[r, s, [np.random.choice(p) for p in parts]]))\n                            if present[s] else np.zeros((3, *cache.shape[3:]), np.uint8)\n                            for s in range(cache.shape[1])])\n        elif self.train:\n            # Slice the memmap BEFORE materialising, so the discarded slices are\n            # never read off disk at all (~37% less I/O per sample).\n            avail = cache.shape[2]\n            t = min(T_TRAIN, avail)\n            t0 = np.random.randint(0, avail - t + 1)\n            vol = np.asarray(cache[r][:, t0:t0 + t])\n            vol = np.stack([self._aug(v) if present[s] else v\n                            for s, v in enumerate(vol)])\n        else:\n            vol = np.asarray(cache[r])               # eval uses every slice\n\n        y, w = self.y[j], self.w[j]\n        if self.aux is not None:\n            y = np.concatenate([y, self.aux[j]])\n            w = np.concatenate([w, np.isfinite(self.aux[j]).astype(np.float32)])\n        return (torch.from_numpy(np.array(vol, copy=True, order=\'C\')),\n                torch.from_numpy(present.copy()),\n                torch.from_numpy(y.copy()),\n                torch.from_numpy(w.copy()))\n\n\ndef collate(batch):\n    xs, ps, ys, ws = zip(*batch)\n    return torch.stack(xs), torch.stack(ps), torch.stack(ys), torch.stack(ws)\n\n"""Shared model architecture for training and offline inference."""\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport timm\n\ndef make_neighbours(x):\n    # (B,S,T,H,W) -> (B,S,T,3,H,W) from slices t-1,t,t+1 (edges clamped)\n    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)\n    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)\n    return torch.stack([prev, x, nxt], dim=3)\n\n\nclass AttnPool(nn.Module):\n    # Attention over slices within one plane.\n    def __init__(self, dim, hidden=128):\n        super().__init__()\n        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\n\n    def forward(self, x, mask=None):               # x (B,T,D)\n        a = self.score(x).squeeze(-1)\n        if mask is not None:\n            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)\n        a = a.softmax(dim=-1)\n        return (x * a.unsqueeze(-1)).sum(1), a\n\n\nclass LabelAttnPool(nn.Module):\n    # Per-label attention over planes: each finding picks the view it needs.\n    def __init__(self, dim, n_labels):\n        super().__init__()\n        self.score = nn.Linear(dim, n_labels)\n\n    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)\n        a = self.score(x)                          # (B,S,L)\n        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)\n        a = a.softmax(dim=1)                       # over planes, per label\n        z = torch.einsum("bsd,bsl->bld", x, a)     # (B,L,D)\n        return z, a\n\n\nclass KneeNet(nn.Module):\n    def __init__(self, backbone="tf_efficientnet_b0.ns_jft_in1k", n_out=12, n_planes=3,\n                 pretrained=True, drop=0.3, img=224, normalization="legacy",\n                 encoder_chunk=64):\n        super().__init__()\n        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}\n        self.enc = timm.create_model(backbone, pretrained=pretrained,\n                                     num_classes=0, in_chans=3, **kwargs)\n        self.encoder_chunk = encoder_chunk\n        config = timm.data.resolve_model_data_config(self.enc)\n        mean = config[\'mean\'] if normalization == \'pretrained\' else (0.449,)*3\n        std = config[\'std\'] if normalization == \'pretrained\' else (0.226,)*3\n        self.register_buffer(\'input_mean\', torch.tensor(mean).view(1,3,1,1), persistent=False)\n        self.register_buffer(\'input_std\', torch.tensor(std).view(1,3,1,1), persistent=False)\n        d = self.enc.num_features\n        self.slice_pool = AttnPool(d)\n        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))\n        self.plane_pool = LabelAttnPool(d, n_out)\n        self.drop = nn.Dropout(drop)\n        self.head_w = nn.Parameter(torch.zeros(n_out, d))\n        self.head_b = nn.Parameter(torch.zeros(n_out))\n        nn.init.trunc_normal_(self.head_w, std=0.02)\n\n    def forward(self, x, present):\n        # autocast entered HERE, not in the training loop: it is thread-local, so a\n        # loop-level autocast never reaches a threaded forward. Verified reaching the\n        # encoder as float16.\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            B, S, T, H, W = x.shape\n            x = x.float().div(255.)   # ImageNet grey stats\n            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)\n\n            x3 = (x3 - self.input_mean) / self.input_std\n            # Chunk evaluation to bound memory; training keeps BatchNorm batch statistics intact.\n            if not self.training and self.encoder_chunk and len(x3) > self.encoder_chunk:\n                features = torch.cat([self.enc(part) for part in x3.split(self.encoder_chunk)])\n            else:\n                features = self.enc(x3)\n            f = features.reshape(B * S, T, -1)\n            v, _ = self.slice_pool(f)\n            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)\n\n            m = present.clone()\n            m[~m.any(dim=1)] = True                # guard: never all-masked\n            z, plane_att = self.plane_pool(v, m)\n            z = self.drop(z)\n            logits = (z * self.head_w.unsqueeze(0)).sum(-1) + self.head_b\n        return logits, plane_att\n\n\n\ndef slice_thirds(n):\n    # Lower, middle and upper thirds of the cached slice band.\n    parts = np.array_split(np.arange(n), 3)\n    if min(len(p) for p in parts) == 0:\n        return [np.array([0]), np.array([(n - 1) // 2]), np.array([n - 1])]\n    return parts\n\n\ndef spread_groups(n):\n    # 3-slice inputs that each span the band: one slice from every third. For six cached\n    # slices that is [0, 2, 4] and [1, 3, 5]; a 3-slice input is its own single group.\n    parts = slice_thirds(n)\n    return [[int(p[g]) for p in parts] for g in range(min(len(p) for p in parts))]\n\n\nclass SlotQueryHead(nn.Module):\n    # Per-finding query attention over series slots (public DINOv2 member head).\n    def __init__(self, dim, n_slots, n_out, hidden=256, drop=0.2):\n        super().__init__()\n        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())\n        self.slot_emb = nn.Parameter(torch.randn(n_slots, hidden) * 0.02)\n        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)\n        self.drop = nn.Dropout(drop)\n        self.out = nn.Linear(hidden, n_out)\n        self.hidden = hidden\n\n    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)\n        h = self.proj(x) + self.slot_emb\n        a = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5\n        a = a.masked_fill(~mask.unsqueeze(1), torch.finfo(a.dtype).min).softmax(-1)\n        z = self.drop(torch.einsum("bos,bsh->boh", a, h))\n        return (z * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias, a\n\n\nclass SlotQueryNet(nn.Module):\n    """One 3-slice image per series slot, encoder partly fine-tuned, query head on top.\n\n    Training feeds one spread triplet per slot (T=3). Evaluation feeds every cached slice\n    and averages logits over the spread groups, so no slice position is left unseen.\n    """\n    def __init__(self, backbone="vit_small_patch14_dinov2.lvd142m", n_out=12, n_slots=6,\n                 pretrained=True, img=336, drop=0.2, hidden=256, unfreeze_last=6,\n                 normalization="pretrained", encoder_chunk=48):\n        super().__init__()\n        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}\n        self.enc = timm.create_model(backbone, pretrained=pretrained,\n                                     num_classes=0, in_chans=3, **kwargs)\n        self.img = img\n        self.encoder_chunk = encoder_chunk\n        self.tokens = hasattr(self.enc, "blocks") and hasattr(self.enc, "num_prefix_tokens")\n        config = timm.data.resolve_model_data_config(self.enc)\n        mean = config[\'mean\'] if normalization == \'pretrained\' else (0.449,)*3\n        std = config[\'std\'] if normalization == \'pretrained\' else (0.226,)*3\n        self.register_buffer(\'input_mean\', torch.tensor(mean).view(1,3,1,1), persistent=False)\n        self.register_buffer(\'input_std\', torch.tensor(std).view(1,3,1,1), persistent=False)\n        if unfreeze_last is not None and hasattr(self.enc, "blocks"):\n            # Early blocks stay at their self-supervised weights; only late blocks adapt.\n            for p in self.enc.parameters(): p.requires_grad = False\n            blocks = list(self.enc.blocks)\n            for block in blocks[max(0, len(blocks) - unfreeze_last):]:\n                for p in block.parameters(): p.requires_grad = True\n            for name in ("norm", "fc_norm"):\n                module = getattr(self.enc, name, None)\n                if isinstance(module, nn.Module):\n                    for p in module.parameters(): p.requires_grad = True\n        dim = self.enc.num_features * 2\n        self.head = SlotQueryHead(dim, n_slots, n_out, hidden=hidden, drop=drop)\n\n    def encode(self, x):                           # (N,3,H,W) normalised -> (N,2D)\n        f = self.enc.forward_features(x)\n        if self.tokens:\n            # CLS token plus mean of patch tokens; register tokens are excluded.\n            return torch.cat([f[:, 0], f[:, self.enc.num_prefix_tokens:].mean(1)], dim=1)\n        if f.ndim == 4:\n            return torch.cat([f.mean((-2, -1)), f.amax((-2, -1))], dim=1)\n        return torch.cat([f, f], dim=1)\n\n    def forward(self, x, present):\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            B, S, T, H, W = x.shape\n            m = present.clone()\n            m[~m.any(dim=1)] = True                # guard: never all-masked\n            keep = m.reshape(-1)\n            groups = spread_groups(T)\n            logits, attention = 0., 0.\n            for group in groups:\n                images = x[:, :, group].reshape(B * S, 3, H, W)[keep].float().div(255.)\n                if H != self.img:\n                    images = nn.functional.interpolate(images, size=(self.img, self.img),\n                                                       mode="bilinear", align_corners=False)\n                images = (images - self.input_mean) / self.input_std\n                # Absent slots are never encoded; their features stay zero and masked.\n                if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:\n                    f = torch.cat([self.encode(part) for part in images.split(self.encoder_chunk)])\n                else:\n                    f = self.encode(images)\n                features = f.new_zeros(B * S, f.shape[-1])\n                features[keep] = f\n                z, a = self.head(features.reshape(B, S, -1), m)\n                logits, attention = logits + z, attention + a\n        return logits / len(groups), attention / len(groups)\n\n\nclass Reader25D(nn.Module):\n    """2.5D study reader (public ConvNeXt reader layout).\n\n    Every cached window of three adjacent slices is one CNN image. Its feature becomes a\n    token tagged with its series slot and its position along the stack; a small\n    transformer mixes all tokens of the study and each finding pools them with its own\n    attention. Training augments each series with one shared rigid/intensity jitter (no\n    flips: laterality is already normalised) and encodes a random subset of windows.\n    """\n    def __init__(self, backbone="convnext_tiny.in12k_ft_in1k", n_out=12, n_slots=6,\n                 pretrained=True, img=240, dim=384, depth=2, drop=0.2, drop_path=0.1,\n                 normalization="pretrained", band=(0.06, 0.94), train_windows=6,\n                 encoder_chunk=96, slot_dropout=0.0, aux_dim=0):\n        super().__init__()\n        self.slot_dropout = slot_dropout     # training only: hide whole series (protocols vary by site)\n        # Runtime switches for XLA/TPU training; neither changes the parameters or the checkpoint.\n        # static_shapes: encode a fixed number of windows for every slot (absent ones masked), so\n        # tensor shapes never depend on the data. augment_in_model=False: the data loader already\n        # applied the per-series jitter (augment_series_cpu), so the model only normalises.\n        self.static_shapes = False\n        self.augment_in_model = True\n        # Test-time window subsets (eval only; inference WINDOW_TTA): encode a random subset of this many\n        # windows per series, as in training, instead of all of them. None: every window.\n        self.subset_windows = None\n        kwargs = {"drop_path_rate": drop_path} if drop_path else {}\n        if backbone.startswith("vit_"):\n            kwargs["img_size"] = img\n        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kwargs)\n        self.img, self.band, self.train_windows = img, tuple(band), train_windows\n        self.encoder_chunk = encoder_chunk\n        config = timm.data.resolve_model_data_config(self.enc)\n        mean = config[\'mean\'] if normalization == \'pretrained\' else (0.449,)*3\n        std = config[\'std\'] if normalization == \'pretrained\' else (0.226,)*3\n        self.register_buffer(\'input_mean\', torch.tensor(mean).view(1,3,1,1), persistent=False)\n        self.register_buffer(\'input_std\', torch.tensor(std).view(1,3,1,1), persistent=False)\n        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim),\n                                  nn.LayerNorm(dim))\n        self.slot_emb = nn.Embedding(n_slots, dim)\n        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))\n        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True,\n                                           norm_first=True)\n        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None\n        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                 nn.Linear(256, n_out))\n        self.cls_w = nn.Parameter(torch.randn(n_out, dim) * 0.02)\n        self.cls_b = nn.Parameter(torch.zeros(n_out))\n        # Report-grounded auxiliary head (training only): regresses the study\'s report embedding\n        # from the finding-pooled and the mean study tokens. Its output is kept in last_aux; the\n        # logits and the inference path do not depend on it.\n        self.aux = (nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU(), nn.Linear(dim, aux_dim))\n                    if aux_dim else None)\n        self.last_aux = None\n\n    def _aux(self, h, pad, pooled):\n        if self.aux is None:\n            return\n        keep = (~pad).unsqueeze(-1).to(h.dtype)\n        mean = (h * keep).sum(1) / keep.sum(1).clamp_min(1)\n        self.last_aux = self.aux(torch.cat([pooled.mean(1), mean], -1))\n\n    def views(self, x, series, n_series):\n        # x float [N,3,H,W] in 0..1; one random transform per series in training.\n        if x.shape[-1] != self.img:\n            x = nn.functional.interpolate(x, size=(self.img, self.img), mode="bilinear",\n                                          align_corners=False, antialias=x.shape[-1] > self.img)\n        if self.training and self.augment_in_model:\n            dev = x.device\n            ang = (torch.rand(n_series, device=dev) - .5) * 2 * np.radians(12)\n            sc = 1 + (torch.rand(n_series, device=dev) - .5) * .24\n            tx, ty = ((torch.rand(2, n_series, device=dev) - .5) * .12)\n            theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),\n                                 torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)\n            grid = nn.functional.affine_grid(theta[series], x.shape, align_corners=False)\n            x = nn.functional.grid_sample(x, grid, mode="bilinear", padding_mode="zeros",\n                                          align_corners=False)\n            gamma = torch.exp((torch.rand(n_series, device=dev) - .5) * .5)[series]\n            gain = (1 + (torch.rand(n_series, device=dev) - .5) * .2)[series]\n            x = x.clamp_min(1e-4).pow(gamma.view(-1, 1, 1, 1)) * gain.view(-1, 1, 1, 1)\n        return (x - self.input_mean) / self.input_std\n\n    def forward(self, x, present):\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            B, S, T, H, W = x.shape\n            K = T // 3\n            m = present.clone()\n            if self.training and self.slot_dropout > 0:\n                kept = m & (torch.rand(m.shape, device=m.device) >= self.slot_dropout)\n                # A study that lost every series keeps one of its present series, chosen at random.\n                pick = torch.rand(m.shape, device=m.device).masked_fill(~m, -1).argmax(dim=1)\n                lost = ~kept.any(dim=1) & m.any(dim=1)\n                one = nn.functional.one_hot(pick, m.shape[1]).bool()\n                m = kept | (lost.unsqueeze(1) & one)   # fixed-shape boolean algebra (XLA-safe)\n            m = m | ~m.any(dim=1, keepdim=True)    # guard: never all-masked\n            if self.static_shapes:\n                return self._forward_static(x, m, B, S, K, H, W)\n            tokens = m.unsqueeze(-1).expand(B, S, K).clone()\n            n_win = self.train_windows if self.training else getattr(self, \'subset_windows\', None)\n            if n_win and n_win < K:\n                # Encode a random subset of windows per series: stack augmentation that\n                # also bounds activation memory. Positions stay exact.\n                rank = torch.rand(B, S, K, device=x.device).argsort(-1).argsort(-1)\n                tokens &= rank < n_win\n            keep = tokens.reshape(-1)\n            series = torch.arange(B * S, device=x.device).repeat_interleave(K)[keep]\n            images = x[:, :, :3 * K].reshape(B * S * K, 3, H, W)[keep].float().div(255.)\n            images = self.views(images, series, B * S).contiguous(memory_format=torch.channels_last)\n            if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:\n                f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])\n            else:\n                f = self.enc(images)\n            feat = f.new_zeros(B * S * K, f.shape[-1])\n            feat[keep] = f\n            slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, K)\n            pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)\n            h = (self.proj(feat).view(B, S, K, -1) + self.slot_emb(slot)\n                 + self.pos_emb(pos.view(1, 1, K, 1).to(feat.dtype)))\n            h = h.reshape(B, S * K, -1)\n            pad = ~tokens.reshape(B, S * K)\n            if self.tf is not None:\n                h = self.tf(h, src_key_padding_mask=pad)\n            a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)\n            pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)\n            logits = (pooled * self.cls_w).sum(-1) + self.cls_b\n            self._aux(h, pad, pooled)\n        return logits, a\n\n\ndef _reader_forward_static(self, x, m, B, S, K, H, W):\n    """Reader25D forward with data-independent shapes (XLA/TPU). Every slot encodes the same\n    number of windows; absent or dropped slots are encoded too but masked, exactly as the\n    dynamic path masks them, so outputs agree up to floating-point error."""\n    n_win = self.train_windows if self.training else getattr(self, \'subset_windows\', None)\n    n_win = n_win if n_win and n_win < K else K\n    if n_win < K:\n        idx = torch.rand(B, S, K, device=x.device).argsort(-1)[..., :n_win].sort(-1).values\n    else:\n        idx = torch.arange(K, device=x.device).view(1, 1, K).expand(B, S, K)\n    windows = x[:, :, :3 * K].reshape(B, S, K, 3, H, W)\n    windows = torch.gather(windows, 2, idx.reshape(B, S, n_win, 1, 1, 1).expand(B, S, n_win, 3, H, W))\n    images = windows.reshape(B * S * n_win, 3, H, W).float().div(255.)\n    series = torch.arange(B * S, device=x.device).repeat_interleave(n_win)\n    images = self.views(images, series, B * S)\n    if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:\n        f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])\n    else:\n        f = self.enc(images)\n    tokens = m.unsqueeze(-1).expand(B, S, n_win)\n    slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, n_win)\n    pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)[idx]\n    h = (self.proj(f).view(B, S, n_win, -1) + self.slot_emb(slot)\n         + self.pos_emb(pos.unsqueeze(-1).to(f.dtype)))\n    h = h.reshape(B, S * n_win, -1)\n    pad = ~tokens.reshape(B, S * n_win)\n    if self.tf is not None:\n        h = self.tf(h, src_key_padding_mask=pad)\n    a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)\n    pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)\n    logits = (pooled * self.cls_w).sum(-1) + self.cls_b\n    self._aux(h, pad, pooled)\n    return logits, a\n\n\nReader25D._forward_static = _reader_forward_static\n\n\ndef augment_series_cpu(x, generator=None):\n    """Reader25D\'s training jitter on the CPU, for loaders that feed XLA devices: one rotation\n    (+-12 deg), scale (+-12%), shift (+-6%), gamma and gain per series, shared by all its slices.\n    x: uint8 [S, T, H, W] of one study; returns uint8 of the same shape (values clipped to 0..255)."""\n    S, T, H, W = x.shape\n    rnd = lambda *shape: torch.rand(*shape, generator=generator)\n    ang = (rnd(S) - .5) * 2 * np.radians(12)\n    sc = 1 + (rnd(S) - .5) * .24\n    tx, ty = (rnd(2, S) - .5) * .12\n    theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),\n                         torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)\n    xf = x.float().div(255.)\n    grid = nn.functional.affine_grid(theta, (S, T, H, W), align_corners=False)\n    xf = nn.functional.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n    gamma = torch.exp((rnd(S) - .5) * .5).view(S, 1, 1, 1)\n    gain = (1 + (rnd(S) - .5) * .2).view(S, 1, 1, 1)\n    xf = xf.clamp_min(1e-4).pow(gamma) * gain\n    return (xf.clamp(0, 1) * 255).round().to(torch.uint8)\n\n\ndef window_centres(n_slices):\n    # Centre slice of each cached 3-slice window (v3 layout); plain slices otherwise.\n    return list(range(1, n_slices, 3)) if n_slices % 3 == 0 and n_slices >= 6 else list(range(n_slices))\n\n\nclass RadImageNetHead(nn.Module):\n    """Frozen official RadImageNet ResNet-50 slice encoder + per-finding query head.\n\n    Follows the public RadImageNet arm (E9/E11): every window-centre slice is one\n    grey image scaled to [-1, 1]; its 2048-d global-average feature is a token tagged\n    with slot and stack position; one query per finding cross-attends over all tokens\n    and is fused with the mean token. Only the head trains.\n    """\n    def __init__(self, n_out=12, n_slots=6, n_windows=8, img=224, head_dim=512,\n                 weights_path=None, encoder_chunk=128):\n        super().__init__()\n        from torchvision.models import resnet50\n        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])\n        if weights_path:\n            state = torch.load(weights_path, map_location="cpu", weights_only=True)\n            self.backbone.load_state_dict({k.split("backbone.", 1)[1]: v for k, v in state.items()},\n                                          strict=True)\n        for p in self.backbone.parameters(): p.requires_grad = False\n        self.img, self.n_windows, self.encoder_chunk = img, n_windows, encoder_chunk\n        dim = 2048\n        self.project = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, head_dim), nn.GELU())\n        self.plane = nn.Parameter(torch.randn(n_slots, head_dim) * .01)\n        self.position = nn.Parameter(torch.randn(n_windows, head_dim) * .01)\n        self.query = nn.Parameter(torch.randn(n_out, head_dim) * .02)\n        self.attn = nn.MultiheadAttention(head_dim, 8, dropout=.10, batch_first=True)\n        self.fuse = nn.Sequential(nn.LayerNorm(head_dim * 4), nn.Linear(head_dim * 4, head_dim),\n                                  nn.GELU(), nn.Dropout(.15))\n        self.weight = nn.Parameter(torch.randn(n_out, head_dim) * .02)\n        self.bias = nn.Parameter(torch.zeros(n_out))\n\n    def train(self, mode=True):\n        super().train(mode)\n        self.backbone.eval()                       # frozen BatchNorm statistics\n        return self\n\n    @torch.no_grad()\n    def encode(self, x, present):\n        # x uint8 (B,S,T,H,W) -> tokens (B, S*K, 2048) float16, token mask (B, S*K)\n        B, S, T, H, W = x.shape\n        centres = window_centres(T)[:self.n_windows]\n        K = len(centres)\n        keep = present.unsqueeze(-1).expand(B, S, K).reshape(-1)\n        images = x[:, :, centres].reshape(B * S * K, 1, H, W)[keep].float().div(127.5).sub(1.)\n        if H != self.img:\n            images = nn.functional.interpolate(images, size=(self.img, self.img), mode="bilinear",\n                                               align_corners=False)\n        images = images.expand(-1, 3, -1, -1)\n        with torch.amp.autocast("cuda", enabled=x.is_cuda):\n            f = torch.cat([self.backbone(part).mean(dim=(2, 3)) for part in images.split(self.encoder_chunk)]) \\\n                if len(images) else images.new_zeros(0, 2048)\n        tokens = torch.zeros(B * S * K, 2048, device=x.device, dtype=torch.float16)\n        tokens[keep] = f.to(torch.float16)\n        return tokens.view(B, S * K, 2048), keep.view(B, S * K).float()\n\n    def head(self, feature, mask):\n        n_slots = self.plane.shape[0]\n        token = self.project(feature.float())\n        token = token.view(len(token), n_slots, -1, token.shape[-1])\n        token = token + self.plane[None, :, None] + self.position[None, None, :token.shape[2]]\n        token = token.flatten(1, 2)\n        pad = mask <= 0\n        empty = pad.all(1)\n        if empty.any():\n            pad = pad.clone(); pad[empty, 0] = False\n        query = self.query.unsqueeze(0).expand(len(token), -1, -1)\n        attended = query + self.attn(query, token, token, key_padding_mask=pad, need_weights=False)[0]\n        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)\n        mean = mean.expand(-1, len(self.query), -1)\n        fused = self.fuse(torch.cat([attended, mean, (attended - mean).abs(), attended * mean], -1))\n        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias\n\n    def forward(self, x, present):\n        m = present.clone()\n        m[~m.any(dim=1)] = True\n        return self.head(*self.encode(x, m)), None\n\n\ndef build_model(cfg, n_out, n_planes, pretrained=True):\n    # A checkpoint\'s recipe rebuilds its own architecture; old recipes have no arch key.\n    arch = cfg.get(\'arch\', \'kneenet\')\n    if arch == \'kneenet\':\n        return KneeNet(cfg[\'backbone\'], n_out, n_planes, pretrained=pretrained,\n                       drop=cfg.get(\'drop\', .3), img=cfg[\'img\'],\n                       normalization=cfg.get(\'normalization\', \'legacy\'))\n    if arch == \'slot_query\':\n        return SlotQueryNet(cfg[\'backbone\'], n_out, n_planes, pretrained=pretrained,\n                            img=cfg[\'img\'], drop=cfg.get(\'drop\', .2),\n                            hidden=cfg.get(\'head_hidden\', 256),\n                            unfreeze_last=cfg.get(\'unfreeze_last\', 6),\n                            normalization=cfg.get(\'normalization\', \'pretrained\'))\n    if arch == \'reader25d\':\n        return Reader25D(cfg[\'backbone\'], n_out, n_planes, pretrained=pretrained,\n                         img=cfg[\'img\'], drop=cfg.get(\'drop\', .2),\n                         dim=cfg.get(\'head_hidden\', 384), drop_path=cfg.get(\'drop_path\', .1),\n                         band=tuple(cfg[\'preprocessing\'][\'slice_band\']),\n                         train_windows=cfg.get(\'train_windows\', 6),\n                         normalization=cfg.get(\'normalization\', \'pretrained\'),\n                         slot_dropout=cfg.get(\'slot_dropout\', 0.0), aux_dim=cfg.get(\'aux_dim\', 0))\n    if arch == \'radhead\':\n        # The frozen encoder ships inside every checkpoint; weights_path is training-only.\n        return RadImageNetHead(n_out, n_planes, n_windows=cfg.get(\'n_windows\', 8), img=cfg[\'img\'],\n                               head_dim=cfg.get(\'head_hidden\', 512),\n                               weights_path=cfg.get(\'weights_path\') if pretrained else None)\n    raise ValueError(f\'Unknown architecture {arch!r}\')\n\ndef masked_bce(logits, targets, weights=None):\n    # BCE over (B,12) soft targets that may contain NaN. NaN cells get exactly zero\n    # gradient, so a study a reader abstained on still trains the other heads. Optional\n    # per-cell weights up-weight gold rows; the mean is taken over observed weight.\n    mask = ~torch.isnan(targets)\n    if not mask.any():\n        return logits.sum() * 0.0\n    w = mask.float() if weights is None else weights * mask\n    loss = F.binary_cross_entropy_with_logits(\n        logits, torch.nan_to_num(targets, nan=0.0), reduction="none")\n    return (loss * w).sum() / w.sum().clamp_min(1e-6)\n\n"""RadImageNet head family: encode every study once, then train one head per fold.\n\nEmbedded in the RadImageNet training notebook in place of train_runtime.py. The frozen\nencoder never changes, so its tokens are computed a single time; each fold then trains\nonly the query head, following the public E9/E11 loop (AdamW 2e-4, weight decay 3e-3,\nbatch 48, at most 24 epochs, early stop after 5 epochs without a pseudo-validation gain).\n"""\nimport hashlib, importlib.metadata\n\nRAD_SHA256 = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"\nRAD_PATIENCE = 5\nRAD_VERIFY = globals().get(\'RAD_VERIFY\', True)   # official checkpoint hash check\n\n\ndef find_radimagenet(root):\n    hits = []\n    for path, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in (\'train_series\', \'test_series\')]\n        hits += [os.path.join(path, f) for f in files if f == \'ResNet50.pt\']\n    assert len(hits) == 1, f\'Attach the official RadImageNet ResNet50.pt exactly once (found {hits})\'\n    digest = hashlib.sha256(open(hits[0], \'rb\').read()).hexdigest()\n    assert digest == RAD_SHA256 or not RAD_VERIFY, f\'RadImageNet checkpoint differs: {digest}\'\n    return hits[0], digest\n\n\nclass CentreSlices(Dataset):\n    # Reads only the window-centre slices the encoder uses (a third of the cache).\n    def __init__(self, frame, centres):\n        self.frame, self.centres, self.caches = frame.reset_index(drop=True), centres, {}\n        self.present = self.frame[[f"present_{s}" for s in range(len(PLANES))]].to_numpy(bool)\n    def __len__(self): return len(self.frame)\n    def __getitem__(self, i):\n        shard = int(self.frame.at[i, \'cache_id\']) if \'cache_id\' in self.frame else 0\n        if shard not in self.caches:\n            self.caches[shard] = np.load(os.path.join(CACHE_DIRS[shard], "cache.npy"), mmap_mode="r")\n        vol = np.asarray(self.caches[shard][int(self.frame.at[i, \'row\'])][:, self.centres])\n        return torch.from_numpy(np.ascontiguousarray(vol)), torch.from_numpy(self.present[i].copy())\n\n\nos.makedirs(OUT, exist_ok=True)\nfolds = assign_folds(df, LABEL_COLS, N_FOLDS, SEED, GROUP_COL if GROUP_COL in df else None)\nfold_table = df[[\'StudyInstanceUID\', \'source\']].assign(fold=folds)\nwrite_atomic(os.path.join(OUT, \'folds.csv\'), lambda p: fold_table.to_csv(p, index=False))\nweights_path, weights_sha = find_radimagenet(LABEL_ROOT)\ncentres = window_centres(N_SLICES)\nidentity = hashlib.sha256((df.to_json(orient=\'split\') + json.dumps(CACHE_META, sort_keys=True)\n                           + fold_table.to_json()).encode()).hexdigest()\nrecipe = dict(arch=\'radhead\', backbone=\'radimagenet_resnet50\', preset=PRESET, img=IMG, n_slices=N_SLICES, planes=PLANES,\n              label_cols=LABEL_COLS, preprocessing=CACHE_META, head_hidden=HEAD_HIDDEN,\n              n_windows=len(centres), seed=SEED, epochs=EPOCHS, lr=LR, weight_decay=WD, batch=BATCH,\n              clip=CLIP, patience=RAD_PATIENCE, n_folds=N_FOLDS, gold_weight=GOLD_WEIGHT,\n              group_col=GROUP_COL if GROUP_COL in df else None, encoder_sha256=weights_sha,\n              identity=identity, versions={p: importlib.metadata.version(p) for p in (\'torch\',\'timm\',\'numpy\')})\nwrite_atomic(os.path.join(OUT, \'run_config.json\'), json.dumps(recipe, indent=2))\n\n# ---- encode once ----------------------------------------------------------------\ntorch.manual_seed(SEED)\nencoder = build_model(dict(recipe, weights_path=weights_path), len(LABEL_COLS), len(PLANES)).to(DEV).eval()\ntokens = np.zeros((len(df), len(PLANES) * len(centres), 2048), np.float16)\nmasks = np.zeros((len(df), len(PLANES) * len(centres)), np.float32)\nloader = DataLoader(CentreSlices(df, centres), batch_size=8, num_workers=NUM_WORKERS, shuffle=False)\nstarted, offset = time.time(), 0\nfor x, present in tqdm(loader, desc=\'RadImageNet encode\'):\n    # The encoder expects window layout (centre at 3k+1); rebuild it from the centre slices.\n    full = torch.zeros((*x.shape[:2], 3 * len(centres), *x.shape[3:]), dtype=torch.uint8)\n    full[:, :, [3 * k + 1 for k in range(len(centres))]] = x\n    t, m = encoder.encode(full.to(DEV), present.to(DEV))\n    tokens[offset:offset + len(x)], masks[offset:offset + len(x)] = t.cpu().numpy(), m.cpu().numpy()\n    offset += len(x)\nprint(f\'encoded {offset:,} studies in {(time.time() - started) / 60:.1f} min\')\nbackbone_state = {k: v for k, v in encoder.state_dict().items() if k.startswith(\'backbone.\')}\ny_all = df[LABEL_COLS].to_numpy(np.float32)\nw_all = df[[f\'w_{c}\' for c in LABEL_COLS]].to_numpy(np.float32) if f\'w_{LABEL_COLS[0]}\' in df else np.ones_like(y_all)\n\n\ndef predict_tokens(model, rows):\n    model.eval(); out = []\n    with torch.inference_mode():\n        for b in range(0, len(rows), 256):\n            r = rows[b:b + 256]\n            logits = model.head(torch.from_numpy(tokens[r]).to(DEV), torch.from_numpy(masks[r]).to(DEV))\n            out.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(out) if out else np.empty((0, len(LABEL_COLS)))\n\n\nfor fold in FOLDS_TO_RUN:\n    torch.manual_seed(SEED + 100 + fold); np.random.seed(SEED + 100 + fold)\n    tr_rows = np.flatnonzero(((folds >= 0) & (folds != fold)) | ((folds == -1) & (GOLD_WEIGHT > 0)))\n    va_rows = np.flatnonzero(folds == fold)\n    assert not set(tr_rows) & set(va_rows) and df.iloc[va_rows][\'source\'].eq(\'pseudo\').all()\n    model = build_model(recipe, len(LABEL_COLS), len(PLANES), pretrained=False).to(DEV)\n    model.load_state_dict({**model.state_dict(), **backbone_state})\n    head_params = [p for n, p in model.named_parameters() if not n.startswith(\'backbone.\')]\n    opt = torch.optim.AdamW(head_params, lr=LR, weight_decay=WD)\n    best, best_state, stale, hist = -1., None, 0, []\n    for ep in range(EPOCHS):\n        model.train(); t0, total = time.time(), 0.\n        order = np.random.permutation(tr_rows)\n        for b in range(0, len(order), BATCH):\n            r = order[b:b + BATCH]\n            feature, mask = torch.from_numpy(tokens[r]).to(DEV), torch.from_numpy(masks[r]).to(DEV)\n            y, w = torch.from_numpy(y_all[r]).to(DEV), torch.from_numpy(w_all[r]).to(DEV)\n            loss = masked_bce(model.head(feature, mask).float(), y, w)\n            opt.zero_grad(set_to_none=True); loss.backward()\n            torch.nn.utils.clip_grad_norm_(head_params, CLIP); opt.step()\n            total += loss.item() * len(r)\n        pv = predict_tokens(model, va_rows)\n        auc, _ = macro_auc(y_all[va_rows], pv, LABEL_COLS)\n        hist.append(dict(fold=fold, epoch=ep, loss=total / len(tr_rows), auc_pseudo=auc, minutes=(time.time() - t0) / 60))\n        print(hist[-1])\n        # Pseudo-label validation AUC selects the epoch; gold is never consulted.\n        if auc > best + 2e-4:\n            best, stale = auc, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n            if stale >= RAD_PATIENCE: break\n    model.load_state_dict(best_state)\n    torch.save(dict(model=best_state, cfg=recipe, epoch=int(np.argmax([h[\'auc_pseudo\'] for h in hist])),\n                    fold=fold, auc_pseudo=best), os.path.join(OUT, f\'fold_{fold}.pt\'))\n    pd.DataFrame(hist).to_csv(os.path.join(OUT, f\'history_fold_{fold}.csv\'), index=False)\n    oof = pd.DataFrame(predict_tokens(model, va_rows), columns=LABEL_COLS)\n    oof.insert(0, \'StudyInstanceUID\', df.iloc[va_rows][\'StudyInstanceUID\'].to_numpy()); oof[\'fold\'] = fold\n    oof.to_csv(os.path.join(OUT, f\'oof_fold_{fold}.csv\'), index=False)\n    gold_rows = np.flatnonzero(folds == -1)\n    pg = predict_tokens(model, gold_rows)\n    gold_output = pd.DataFrame(pg, columns=LABEL_COLS)\n    gold_output.insert(0, \'StudyInstanceUID\', df.iloc[gold_rows][\'StudyInstanceUID\'].to_numpy())\n    gold_output.to_csv(os.path.join(OUT, f\'gold_fold_{fold}.csv\'), index=False)\n    print(\'Gold audit\' + (\' (IN-SAMPLE: gold trained at weight %g)\' % GOLD_WEIGHT if GOLD_WEIGHT > 0 else \'\') + \':\',\n          macro_auc(df.iloc[gold_rows][LABEL_COLS].to_numpy(), pg, LABEL_COLS)[0])\n    del model, opt\n    gc.collect()\n    if DEV == \'cuda\': torch.cuda.empty_cache()\n\npaths = [os.path.join(OUT, f\'oof_fold_{f}.csv\') for f in range(N_FOLDS)]\nif all(os.path.exists(p) for p in paths):\n    oof = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)\n    expected = df.loc[folds >= 0, \'StudyInstanceUID\']\n    aligned = oof.set_index(\'StudyInstanceUID\').loc[expected]\n    oof.to_csv(os.path.join(OUT, \'oof_predictions.csv\'), index=False)\n    print(\'OOF pseudo AUC (noisy-label proxy, selection-biased):\',\n          macro_auc(df.loc[folds >= 0, LABEL_COLS].to_numpy(), aligned[LABEL_COLS].to_numpy(), LABEL_COLS))\n'


In [ ]:
"""Two-GPU training session: one queue of jobs per GPU, each job its own process.

Embedded in the session notebooks. Kaggle counts GPU quota per session hour, and its T4
sessions come with two GPUs, so two independent trainings run side by side for the
price of one. These are separate processes pinned with CUDA_VISIBLE_DEVICES, not
DataParallel: every job is exactly the single-GPU recipe of the per-fold notebooks.
Expects GPU_JOBS, RESUME_DIR, TRAIN_JOB_SOURCE and RAD_JOB_SOURCE from earlier cells.
A job is (preset, folds) or (preset, folds, seed); a non-default seed trains another member
of the same family in its own folder, <preset>_s<seed>.
"""
import json, os, subprocess, sys, threading, time
from pathlib import Path

WORK = Path(globals().get('WORK', '/kaggle/working'))
POLL_SECONDS = globals().get('POLL_SECONDS', 120)
WORKERS_PER_JOB = globals().get('WORKERS_PER_JOB', 2)      # 4 CPU cores shared by two jobs
REQUIRE_GPUS = globals().get('REQUIRE_GPUS', True)
KEYWORDS = ('recipe:', 'label readers', 'trainable', "'epoch'", 'Gold audit', 'OOF pseudo',
            'encoded', 'Error', 'Traceback', 'Killed')

jobs_dir, logs_dir = WORK / '_jobs', WORK / 'logs'
jobs_dir.mkdir(parents=True, exist_ok=True); logs_dir.mkdir(exist_ok=True)
(jobs_dir / 'train_job.py').write_text(TRAIN_JOB_SOURCE)
(jobs_dir / 'rad_job.py').write_text(RAD_JOB_SOURCE)
if REQUIRE_GPUS:
    import torch
    assert torch.cuda.device_count() >= len(GPU_JOBS), \
        f'This session runs {len(GPU_JOBS)} GPU queues: set Accelerator to GPU T4 x2 (found {torch.cuda.device_count()})'
def job_parts(job):
    preset, folds, *rest = job
    seed = rest[0] if rest else None
    return preset, folds, seed, (preset if seed is None else f'{preset}_s{seed}')


seen = [(name, f) for queue in GPU_JOBS.values() for job in queue
        for _, folds, _, name in [job_parts(job)] for f in folds]
assert len(seen) == len(set(seen)), f'A family-fold appears twice in GPU_JOBS: {GPU_JOBS}'

results = {}
low_ram = {}


def available_ram_gb():
    # Cache pages of the memory-mapped shards are reclaimable and counted as available;
    # this tracks what the jobs themselves hold. None where /proc is missing (local tests).
    try:
        for line in open('/proc/meminfo'):
            if line.startswith('MemAvailable:'):
                return int(line.split()[1]) / 2**20
    except OSError:
        return None


def run_queue(gpu, queue):
    for job in queue:
        preset, folds, seed, run_name = job_parts(job)
        out = WORK / run_name
        out.mkdir(exist_ok=True)
        name = f"gpu{gpu}_{run_name}_f{''.join(map(str, folds))}"
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), KNEE_PRESET=preset, KNEE_RUN_NAME=run_name,
                   KNEE_FOLDS=json.dumps(folds), KNEE_OUT=str(out), KNEE_WORKERS=str(WORKERS_PER_JOB),
                   KNEE_RESUME_DIR=RESUME_DIR, TQDM_DISABLE='1', PYTHONUNBUFFERED='1')
        if seed is not None:
            env['KNEE_SEED'] = str(seed)
        script = jobs_dir / ('rad_job.py' if preset == 'radimagenet' else 'train_job.py')
        started = time.time()
        with open(logs_dir / f'{name}.log', 'w') as handle:
            code = subprocess.run([sys.executable, str(script)], env=env, stdout=handle,
                                  stderr=subprocess.STDOUT).returncode
        results[name] = dict(returncode=code, minutes=round((time.time() - started) / 60, 1))
        print(f'[{name}] finished with code {code} after {results[name]["minutes"]} min', flush=True)
        if code != 0:
            break       # later jobs on this GPU are skipped; the failure is reported below


def show_new_lines(offsets):
    for log in sorted(logs_dir.glob('*.log')):
        lines = log.read_text(errors='replace').splitlines()
        for line in lines[offsets.get(log, 0):]:
            if any(k in line for k in KEYWORDS):
                print(f'[{log.stem}] {line[:400]}', flush=True)
        offsets[log] = len(lines)


session_start = time.time()
threads = [threading.Thread(target=run_queue, args=(gpu, queue), daemon=True) for gpu, queue in GPU_JOBS.items()]
for thread in threads: thread.start()
offsets = {}
while any(thread.is_alive() for thread in threads):
    time.sleep(POLL_SECONDS)
    show_new_lines(offsets)
    ram = available_ram_gb()
    if ram is not None and ram < low_ram.get('gb', float('inf')):
        low_ram.update(gb=round(ram, 1), minute=round((time.time() - session_start) / 60))
        print(f'[ram] lowest available so far: {ram:.1f} GB', flush=True)
for thread in threads: thread.join()
show_new_lines(offsets)
print(json.dumps(results, indent=2))
if low_ram:
    print(f"Lowest available RAM: {low_ram['gb']} GB (minute {low_ram['minute']})")
(WORK / 'session_results.json').write_text(json.dumps(dict(jobs=results, lowest_available_ram=low_ram), indent=2))
expected = sum(len(queue) for queue in GPU_JOBS.values())
failed = {k: v for k, v in results.items() if v['returncode'] != 0}
for name in failed:
    print(f'\n===== tail of {name} =====\n' + '\n'.join((logs_dir / f'{name}.log').read_text(errors='replace').splitlines()[-40:]))
assert not failed and len(results) == expected, f'Jobs failed or were skipped: {failed or "see above"}'
print('All jobs finished. Outputs: ' + ', '.join(sorted({job_parts(j)[3] for q in GPU_JOBS.values() for j in q})))
